# V27 Signature Ensemble — Stable Prompt Groups

In [ ]:
'''Credits and scope
Adapted from Dhruv Tibarewal, nlm-v27: https://www.kaggle.com/code/dhruvtibarewal/nlm-v27
The user reports a public score of 0.651 for the source. That score is not measured for this adaptation.
This ungated variant omits the source Llama base and its six adapters while access approval is pending.
All remaining model weights and both research policies stay fixed; removing a member can change the score.
Upstream components credited in the source: Nishant Dahal (Part Two checkpoints), prvsiyan,
lucasboesen, vnton37nguyn, kirilltym and sangam21 (frozen features, Gaussian heads and LoRA readouts),
and Joseph Adamski (the public ATLAS bundle). Their original numerical recipes are retained.

Pipeline: 12 v5 adapters (five folds plus full-data for each of Qwen3-14B and Qwen3-8B),
two Part Two 14B adapters, fine-tuned DeBERTa, four-view SVC on pool v5, prompt-aware heads,
15% complementary frozen-feature ensemble, selective 10% ATLAS, Sinkhorn and source postprocessing.
Qwen3.5 token attention stays disabled, exactly as in the supplied v27 run.

Refactor differences: real class methods and explicit object state; exact complete-weight checks;
scoped dense-PEFT compatibility; checked child processes instead of unchecked shell pipelines;
no fitting of the source TF-IDF classifier when its weight is zero (its evidence was identically zero).
The source permits unavailable optional legacy LoRA members and budget-limited ATLAS to be skipped.
Every such skip is retained in worker logs and stage_timings.json; a failed complementary process fails the run.

Validation limit: the supplied saved run took about 87 minutes on a three-row test preview.
That is not a hidden-test runtime estimate. Full GPU parity and public score require Kaggle execution.
Published backbone training includes official rows: in-sample checks are reproduction checks, not OOF scores.
'''

In [ ]:
'''Research hypothesis
A mistaken prompt group contaminates centered heads, joint Gaussian inference and dense-group Hungarian assignment.
Keep a main-ensemble group only when its complete row membership is identical at cosine-similarity cuts
0.12, 0.15 and 0.18 (the source uses 0.15). Unstable groups become independent rows before feature readout.
This removes group evidence without suppressing the plain model evidence. Complementary grouping stays unchanged.
All checkpoints, fold counts, ensemble weights, typography rules and ATLAS settings stay fixed.
Two extra CPU clustering passes are required; there are no additional GPU feature passes.
A genuine group near a cut may be rejected, so this experiment can lower the score. No gain is guaranteed.
'''

In [ ]:
'''Pinned Kaggle inputs are embedded in metadata.kaggle.dataSources below.
After importing, verify the Inputs panel before running. These notebooks use the nlm-v27 inputs except the gated Llama model.
No spectral bank from iterations 13/14 is used here.
Dataset sources:
https://www.kaggle.com/datasets/dhruvtibarewal/nlm-l24-adapters
https://www.kaggle.com/datasets/seyitkaangunes/ntl-extra-api-passages
https://www.kaggle.com/datasets/abdullah1afify/ntl-generations
https://www.kaggle.com/datasets/dophuchung/ntl-proxy-generations
https://www.kaggle.com/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head
https://www.kaggle.com/datasets/dhruvtibarewal/nlm-matched-clean
https://www.kaggle.com/datasets/dhruvtibarewal/nlm-offline-wheels
https://www.kaggle.com/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1
https://www.kaggle.com/datasets/lucasboesen/ntl-free-proxy-generations
https://www.kaggle.com/datasets/nishantdahal/ntl-two-stage-lora
https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend
https://www.kaggle.com/datasets/dhruvtibarewal/nlm-l24-adapters-v5
https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset
https://www.kaggle.com/datasets/pjmathematician/ultrabert-380m-ntl
Notebook sources:
https://www.kaggle.com/code/lucasboesen/name-that-llm-glm4-9b-trunc25
https://www.kaggle.com/code/lucasboesen/name-that-llm-extract-encoders2
https://www.kaggle.com/code/lucasboesen/name-that-llm-electra-large-weights
https://www.kaggle.com/code/lucasboesen/name-that-llm-desklib-weights
https://www.kaggle.com/code/lucasboesen/name-that-llm-train-lora24-s7
https://www.kaggle.com/code/lucasboesen/name-that-llm-train-tokattn
https://www.kaggle.com/code/lucasboesen/name-that-llm-train-tokattn2
https://www.kaggle.com/code/vnton37nguyn/ntl-train-lora-qwen25-instyle
Model sources:
https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/4b-base/1
https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/14b/1
https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/8b-base/1
https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/8b/1
https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/14b-base/1
https://www.kaggle.com/models/qwen-lm/qwen2.5/transformers/7b/1
https://www.kaggle.com/models/qwen-lm/qwen2.5/transformers/14b/1
Competition: https://www.kaggle.com/competitions/name-that-llm
'''

In [ ]:
import os, sys, json, time, glob, gc, re, pathlib, hashlib, importlib, importlib.util, subprocess, multiprocessing
from pathlib import Path
from dataclasses import dataclass
from contextlib import contextmanager, redirect_stdout, redirect_stderr
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from threadpoolctl import threadpool_limits
SOURCE_URL = "https://www.kaggle.com/code/dhruvtibarewal/nlm-v27"
SOURCE_SHA256 = 'e20eb6183b3544231e73d86bf63949248b1c664abbfcc3e5241081651af47f5d'
LABELS = ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max']


In [ ]:
'''Runtime configuration. Both experiments keep the source model weights and 90/10 ATLAS rule.
The research version changes only main-ensemble prompt grouping. No model training runs here.'''


In [ ]:
class ExperimentConfig:
    input_root: Path = Path('/kaggle/input')
    output_root: Path = Path('/kaggle/working')
    run_inference: bool = True
    stable_groups: bool = True
    group_delta: float = 0.03
    total_budget_seconds: int = 11 * 3600


In [ ]:
class RuntimeCompatibility:
    @staticmethod
    @contextmanager
    def dense_peft():
        '''Dense LoRA does not use torchao; patch every imported availability alias temporarily.'''
        original_find = importlib.util.find_spec
        patched = []
        unavailable = lambda *args, **kwargs: False
        try:
            importlib.util.find_spec = lambda name, *args, **kwargs: None if name.split('.')[0] == 'torchao' else original_find(name, *args, **kwargs)
            peft = importlib.import_module('peft')
            utilities = importlib.import_module('peft.import_utils')
            original_check = utilities.is_torchao_available
            for name in ['peft.import_utils', 'peft.tuners.lora.model', 'peft.tuners.lora.torchao']:
                module = importlib.import_module(name)
                if hasattr(module, 'is_torchao_available'):
                    previous = module.is_torchao_available
                    patched.append((module, original_check if previous is unavailable else previous))
                    module.is_torchao_available = unavailable
            yield peft
        finally:
            for module, previous in reversed(patched):
                module.is_torchao_available = previous
            importlib.util.find_spec = original_find

    @classmethod
    def probe(cls):
        with cls.dense_peft() as peft:
            base = torch.nn.Sequential(torch.nn.Linear(4, 4, bias=False, device='cpu'))
            adapter = peft.get_peft_model(base, peft.LoraConfig(r=2, lora_alpha=4, target_modules=['0'], lora_dropout=0.0, bias='none')).eval()
            with torch.inference_mode():
                result = adapter(torch.ones(1, 4, device='cpu'))
            if tuple(result.shape) != (1, 4) or not bool(torch.isfinite(result).all()):
                raise RuntimeError('Dense PEFT CPU probe failed')
        print('PEFT DENSE-LORA CPU PROBE PASSED', flush=True)


In [ ]:
class InputAssets:
    def __init__(self, config):
        self.config = config
        self.files = sorted(config.input_root.rglob('*'))
        self.configs = [p for p in self.files if p.name == 'config.json']
        self.records = []

    @staticmethod
    def has_weights(directory):
        directory = Path(directory)
        for name in ['model.safetensors', 'pytorch_model.bin']:
            p = directory / name
            if p.is_file() and p.stat().st_size > 0:
                return True
        for name in ['model.safetensors.index.json', 'pytorch_model.bin.index.json']:
            p = directory / name
            if p.is_file():
                shards = set(json.loads(p.read_text()).get('weight_map', {}).values())
                if shards and all((directory / s).is_file() and (directory / s).stat().st_size > 0 for s in shards):
                    return True
        return False

    def unique(self, suffix):
        hits = [p for p in self.files if str(p).endswith('/' + suffix)]
        if len(hits) != 1:
            raise FileNotFoundError(f'Expected exactly one {suffix}; found {len(hits)}. Check Kaggle Inputs and duplicate versions.')
        return hits[0]

    def base(self, *parts):
        hits = [p.parent for p in self.configs if all(part in str(p).lower() for part in parts) and self.has_weights(p.parent)]
        if len(hits) != 1:
            raise FileNotFoundError(f'Expected one complete base {parts}; found {len(hits)}. A base_config directory has no weights.')
        self.records.append({'kind': 'base', 'path': str(hits[0])})
        return str(hits[0])

    def adapter(self, name):
        root = self.unique(name + '/heads.pt').parent
        ad = root / 'adapter' if (root / 'adapter/adapter_config.json').is_file() else root
        cfg = json.loads((ad / 'adapter_config.json').read_text())
        if not (ad / 'adapter_model.safetensors').is_file() or (ad / 'adapter_model.safetensors').stat().st_size == 0:
            raise FileNotFoundError(f'Incomplete LoRA: {ad}')
        if cfg.get('use_dora') or cfg.get('use_rslora') or cfg.get('rank_pattern') or cfg.get('alpha_pattern'):
            raise ValueError(f'The source merge implementation does not support this adapter configuration: {ad}')
        if int(cfg['r']) <= 0:
            raise ValueError(f'Invalid LoRA rank: {ad}')
        self.records.append({'kind': 'v5_adapter', 'path': str(root)})
        return str(root)

    def part_two(self, name):
        root = self.unique(name + '/meta.json').parent
        meta = json.loads((root / 'meta.json').read_text())
        cfg = meta['cfg']
        labels = meta['labels']
        if len(labels) != 18 or len(set(labels)) != 18 or set(labels) != set(LABELS):
            raise ValueError(f'Unexpected label mapping: {root}')
        if cfg.get('eos_id') != 151643:
            raise ValueError(f'Training EOS differs from the source: {root}')
        for file in ['heads.safetensors', 'adapter/adapter_config.json', 'adapter/adapter_model.safetensors']:
            if not (root / file).is_file():
                raise FileNotFoundError(root / file)
        return str(root), cfg

    def preflight(self):
        data = [p.parent for p in self.files if p.name == 'train.csv' and p.parent.name == 'name-that-llm']
        if len(data) != 1 or not (data[0] / 'test.csv').is_file():
            raise FileNotFoundError('Attach the Name That LLM competition')
        self.data = str(data[0])
        self.b8 = self.base('/qwen-3/', '/8b-base/')
        self.b14 = self.base('/qwen-3/', '/14b-base/')
        self.b25_14 = self.base('/qwen2.5/', '/14b/')
        self.q14 = [self.adapter(f'q14v5_f{k}') for k in range(5)] + [self.adapter('q14v5')]
        self.q8 = [self.adapter(f'q8v5_f{k}') for k in range(5)] + [self.adapter('q8v5')]
        self.n14, self.n14cfg = self.part_two('lora_qwen3_14b')
        self.q25, self.q25cfg = self.part_two('lora_qwen25_14b')
        self.deberta = str(self.unique('deberta_v3_large/meta.json').parent)
        if not self.has_weights(Path(self.deberta) / 'encoder') or not (Path(self.deberta) / 'head.safetensors').is_file():
            raise FileNotFoundError('Incomplete Part Two DeBERTa encoder or head')
        self.pool = str(self.unique('nlm_pool_v5.parquet'))
        self.extra = str(self.unique('nlm_matched_clean.parquet'))
        self.atlas = str(self.unique('atlas_bundle.json').parent)
        if not (Path(self.atlas) / 'adapters.safetensors').is_file():
            raise FileNotFoundError('Incomplete ATLAS bundle')
        train = pd.read_csv(Path(self.data) / 'train.csv')
        test = pd.read_csv(Path(self.data) / 'test.csv')
        if not {'id', 'text', 'model', 'prompt_id'} <= set(train) or not {'id', 'text'} <= set(test):
            raise ValueError('Competition CSV schema differs')
        if set(train.model) != set(LABELS) or not test.id.is_unique or len(test) == 0:
            raise ValueError('Unexpected labels or test IDs')
        pool = pd.read_parquet(self.pool, columns=['text', 'model'])
        if pool.empty or not set(LABELS) <= set(pool.model):
            raise ValueError('The v5 pool does not cover all 18 labels')
        # Validate all frozen backbones before their slow feature extraction starts.
        for kind, hidden, dirname in [('glm4', 4096, None), ('qwen3', 4096, None), ('qwen2', 3584, None), ('qwen3', 2560, None), (None, None, 'deberta-v3-large'), (None, None, 'electra-large-discriminator'), (None, None, 'desklib-detector')]:
            hits = []
            for p in self.configs:
                if not self.has_weights(p.parent):
                    continue
                c = json.loads(p.read_text())
                if (dirname and p.parent.name == dirname) or (not dirname and c.get('model_type') == kind and c.get('hidden_size') == hidden):
                    hits.append(p.parent)
            if not hits:
                raise FileNotFoundError(f'Missing complete complementary backbone: {kind}, {hidden}, {dirname}')
        self.unique('tokattn_debl23_final.pt')
        self.unique('ultrabert-380m-ntl/fold0/config.json')
        manifest = {'source': SOURCE_URL, 'source_sha256': SOURCE_SHA256, 'rows': {'train': len(train), 'test': len(test), 'pool': len(pool)}, 'resolved': self.records, 'stable_groups': self.config.stable_groups, 'omitted_source_members': ['ll8v5'],
                    'optional_public_lora24': [str(p) for p in self.files if str(p).endswith('/lora24/final/COMPLETE.json')],
                    'note': 'The upstream source skips unavailable legacy LoRA members; every skip is recorded in worker logs. Qwen3.5 is disabled exactly as in v27.'}
        self.config.output_root.mkdir(parents=True, exist_ok=True)
        (self.config.output_root / 'input_manifest.json').write_text(json.dumps(manifest, indent=2))
        print(f'INPUT CHECKS PASSED: 12 v5 adapters, Part Two, complete bases, pool v5, ATLAS. Test rows: {len(test)}', flush=True)
        return self

    def models(self):
        layers = [2, 4, 6, 8, 12, 16, 20, 24]
        return [dict(name='q14v5', type='l24', path=self.b14, adapters=self.q14, keep=27, layers=layers, w=3.0, bs=8, max_len=384, twin=0.35, gauss=0.70),
                dict(name='q8v5', type='l24', path=self.b8, adapters=self.q8, keep=24, layers=layers, w=2.0, bs=16, max_len=384, twin=0.35, gauss=0.44),
                dict(name='n14b', type='l24', path=self.b14, adapters=[self.n14], keep=self.n14cfg['keep'], layers=self.n14cfg['layers'], w=2.5, bs=8, max_len=self.n14cfg['max_len'], twin=0.35, no_final_norm=True, eos_id=self.n14cfg['eos_id']),
                dict(name='q25_14', type='l24', path=self.b25_14, adapters=[self.q25], keep=self.q25cfg['keep'], layers=self.q25cfg['layers'], w=1.0, bs=8, max_len=self.q25cfg['max_len'], twin=0.35, no_final_norm=True, eos_id=self.q25cfg['eos_id']),
                dict(name='deb', type='enc', ckpt=self.deberta, w=1.0)]

In [ ]:
class StableGroupPolicy:
    @staticmethod
    def retain(base, lower, upper, cap=18):
        '''Require identical row membership at three cuts; unstable rows become singletons.'''
        base, lower, upper = map(np.asarray, (base, lower, upper))
        if base.ndim != 1 or not (base.shape == lower.shape == upper.shape):
            raise ValueError('Grouping arrays must have identical one-dimensional shapes')
        result = np.arange(len(base), dtype=int)
        next_label = len(base)
        retained = 0
        for group in np.unique(base):
            rows = np.flatnonzero(base == group)
            if not 2 <= len(rows) <= cap:
                continue
            lo = np.flatnonzero(lower == lower[rows[0]])
            hi = np.flatnonzero(upper == upper[rows[0]])
            if np.array_equal(rows, lo) and np.array_equal(rows, hi):
                result[rows] = next_label
                next_label += 1
                retained += len(rows)
        return result, {'rows': len(base), 'retained_group_rows': retained, 'ungrouped_rows': len(base) - retained}

    @classmethod
    def self_check(cls):
        base = np.array([0, 0, 1, 1, 2])
        stable, audit = cls.retain(base, base, base)
        assert stable[0] == stable[1] and stable[2] == stable[3] and audit['retained_group_rows'] == 4
        conservative, audit = cls.retain(base, [0, 1, 2, 2, 3], base)
        assert conservative[0] != conservative[1] and conservative[2] == conservative[3]
        conservative, _ = cls.retain(base, base, [0, 0, 0, 0, 1])
        assert len(np.unique(conservative)) == 5
        empty, _ = cls.retain([], [], [])
        assert len(empty) == 0
        assert isinstance(json.loads(json.dumps(audit))['retained_group_rows'], int)
        print('GROUP POLICY SELF-CHECK PASSED', flush=True)


In [ ]:
class AdapterHeadChecks:
    @staticmethod
    def run(assets):
        for roots, hidden in [(assets.q14, 5120), (assets.q8, 4096)]:
            for root in roots:
                root = Path(root)
                heads = torch.load(root / 'heads.pt', map_location='cpu', weights_only=True)
                for name in ['plain', 'centered']:
                    if tuple(heads[name + '.weight'].shape) != (18, hidden * 8) or tuple(heads[name + '.bias'].shape) != (18,):
                        raise ValueError(f'Invalid {name} head shape: {root}')
                for layer in range(8):
                    for kind in ['weight', 'bias']:
                        if tuple(heads[f'norms.{layer}.{kind}'].shape) != (hidden,):
                            raise ValueError(f'Invalid pooling normalization: {root}')
                if (root / 'meta.json').is_file():
                    labels = json.loads((root / 'meta.json').read_text())['labels']
                    if len(labels) != 18 or set(labels) != set(LABELS):
                        raise ValueError(f'Invalid label mapping: {root}')
        print('ALL 12 V5 HEAD SHAPES AND LABEL MAPPINGS PASSED', flush=True)


In [ ]:
'''Gaussian numerics: source mathematics preserved; class layers separate responsibilities.'''

In [ ]:
class GaussianNumerics:

    def _lse(self, A, axis=None, keepdims=False):
        _owner = self
        m = np.max(A, axis=axis, keepdims=True)
        m = np.where(np.isfinite(m), m, 0.0)
        out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
        if keepdims:
            return out
        return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

    def log_softmax(self, L):
        _owner = self
        return L - _owner._lse(L, axis=1, keepdims=True)

    def _lw_shrinkage(self, X):
        _owner = self
        '''Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered).'''
        from sklearn.covariance import ledoit_wolf_shrinkage
        return float(ledoit_wolf_shrinkage(X, assume_centered=True))

    def _shrunk(self, S, a):
        _owner = self
        d = S.shape[0]
        return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

    def _two_way(self, Z, y, pinv, G, iters=4):
        _owner = self
        '''additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds).'''
        P = np.zeros((G, Z.shape[1]))
        cnt_p = np.bincount(pinv, minlength=G).astype(float)
        cls = np.unique(y)
        mu = np.zeros((_owner.K, Z.shape[1]))
        for _ in range(iters):
            R = Z - P[pinv]
            for c in cls:
                mu[c] = R[y == c].mean(0)
            R = Z - mu[y]
            P = np.zeros_like(P)
            np.add.at(P, pinv, R)
            P /= cnt_p[:, None]
        return (mu, P, cnt_p)

    def fit(self, X, y, prompts, cfg=None):
        _owner = self
        '''fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit).'''
        c = dict(_owner.DEFAULT)
        c.update(cfg or {})
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        N, D = X.shape
        pu, pinv = np.unique(np.asarray(prompts), return_inverse=True)
        G = len(pu)
        mx = X.mean(0)
        sx = X.std(0)
        sx = np.where(sx > 1e-08, sx, 1.0)
        Xs = (X - mx) / sx
        if c['red'] == 'none':
            V = np.eye(D)
        else:
            if c['red'] == 'wpca':
                Pm = np.zeros((G, D))
                np.add.at(Pm, pinv, Xs)
                Pm /= np.bincount(pinv)[:, None]
                R = Xs - Pm[pinv]
            elif c['red'] == 'pca':
                R = Xs
            else:
                raise ValueError(c['red'])
            d = int(min(c['d'], N - G - 1, D))
            if N < D:
                w, U = np.linalg.eigh(R @ R.T)
                o = np.argsort(w)[::-1][:d]
                w, U = (np.clip(w[o], 1e-12, None), U[:, o])
                V = R.T @ U / np.sqrt(w)
            else:
                _, _, Vt = np.linalg.svd(R, full_matrices=False)
                V = Vt[:d].T
        Z = Xs @ V
        d = Z.shape[1]
        mu, P, ng = _owner._two_way(Z, y, pinv, G)
        E = Z - mu[y] - P[pinv]
        dof = max(N - G - len(np.unique(y)) + 1, 1)
        Se_raw = E.T @ E / dof
        ae = _owner._lw_shrinkage(E) if c['se_shrink'] == 'lw' else float(c['se_shrink'])
        Se = _owner._shrunk(Se_raw, ae)
        Pc = P - P.mean(0)
        Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
        ap = _owner._lw_shrinkage(Pc) if c['sp_shrink'] == 'lw' else float(c['sp_shrink'])
        tr = max(np.trace(Sp_raw), 1e-08 * np.trace(Se))
        Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)
        Sp *= c['sp_scale']
        L = np.linalg.cholesky(Se)
        Li = np.linalg.inv(L)
        M = Li @ Sp @ Li.T
        M = 0.5 * (M + M.T)
        lam, Q = np.linalg.eigh(M)
        lam = np.clip(lam, c['lam_floor'], None)
        A = Q.T @ Li
        W = V @ A.T
        return {'mx': mx, 'sx': sx, 'W': W, 'm': mu @ A.T, 'lam': lam, 'T1': 1.0, 'Tg': 1.0, 'info': {'d': int(d), 'a_e': round(ae, 4), 'a_p': round(ap, 4), 'lam_pct': np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, 'cfg': c}

    def transform(self, X, prm):
        _owner = self
        return (np.asarray(X, dtype=np.float64) - prm['mx']) / prm['sx'] @ prm['W']

    def _inj_mask(self, n, C):
        _owner = self
        if (n, C) not in _owner._MASK:
            grids = np.meshgrid(*[np.arange(C)] * n, indexing='ij')
            bad = np.zeros([C] * n, bool)
            for i, j in itertools.combinations(range(n), 2):
                bad |= grids[i] == grids[j]
            _owner._MASK[n, C] = bad
        return _owner._MASK[n, C]

    def singleton_ll(self, U, m, lam):
        _owner = self
        '''(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out).'''
        R = U[:, None, :] - m[None]
        return -0.5 * (R ** 2 / (1 + lam)).sum(-1)


In [ ]:
class SharedPromptGaussian(GaussianNumerics):

    def joint_terms(self, U, m, lam):
        _owner = self
        '''a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped).'''
        n = len(U)
        gam = lam / (1 + n * lam)
        R = U[:, None, :] - m[None]
        a = -0.5 * (R ** 2 * (1 - gam)).sum(-1)
        G = R * np.sqrt(gam)
        B = np.einsum('icd,jed->ijce', G, G)
        return (a, B)

    def joint_table(self, a, B):
        _owner = self
        '''full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij.'''
        n, C = a.shape
        Q = np.zeros([C] * n)
        for i in range(n):
            sh = [1] * n
            sh[i] = C
            Q = Q + a[i].reshape(sh)
        for i, j in itertools.combinations(range(n), 2):
            sh = [1] * n
            sh[i] = C
            sh[j] = C
            Q = Q + B[i, j].reshape(sh)
        return Q

    def exact_marginals(self, Q, T=1.0, inj=True):
        _owner = self
        '''per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T.'''
        n = Q.ndim
        C = Q.shape[0]
        Qt = Q / T
        if inj and n >= 2:
            Qt = np.where(_owner._inj_mask(n, C), -np.inf, Qt)
        lz = _owner._lse(Qt)
        out = np.empty((n, C))
        for i in range(n):
            ax = tuple((k for k in range(n) if k != i))
            out[i] = (_owner._lse(Qt, axis=ax) if ax else Qt) - lz
        return out

    def em_marginals(self, U, m, lam, T=1.0, iters=5, repel=False, q0=None):
        _owner = self
        '''cavity EM for one group (any n >= 2): returns (n, C) log q.'''
        n = len(U)
        v = lam / (1 + (n - 1) * lam)
        L = _owner.singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
        q = np.exp(_owner.log_softmax(L))
        for _ in range(iters):
            s = U - q @ m
            S = s.sum(0)
            Pm = (S[None] - s) * v
            R = U[:, None, :] - Pm[:, None, :] - m[None]
            L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
            if repel:
                lo = np.log(np.clip(1 - q, 1e-06, None))
                L = L + (lo.sum(0)[None] - lo)
            q = np.exp(_owner.log_softmax(L))
        return _owner.log_softmax(L)

    def lam_g(self, prm, cfg=None):
        _owner = self
        '''lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly.'''
        c = dict(prm.get('cfg', _owner.DEFAULT))
        c.update(cfg or {})
        return np.full_like(prm['lam'], _owner.LAM_DIFF) if c.get('diff') else prm['lam']

    def group_logpost(self, U, prm, T1=None, Tg=None, cfg=None):
        _owner = self
        c = dict(prm.get('cfg', _owner.DEFAULT))
        c.update(cfg or {})
        n = len(U)
        lg = _owner.lam_g(prm, cfg)
        T1 = prm['T1'] if T1 is None else T1
        Tg = prm['Tg'] if Tg is None else Tg
        if n == 1:
            return _owner.log_softmax(_owner.singleton_ll(U, prm['m'], prm['lam']) / T1)
        if n <= c['n_exact']:
            a, B = _owner.joint_terms(U, prm['m'], lg)
            return _owner.exact_marginals(_owner.joint_table(a, B), Tg, c['inj'])
        return _owner.em_marginals(U, prm['m'], lg, Tg, c['iters'], c['repel'])

    def scores(self, X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
        _owner = self
        '''(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed).'''
        U = _owner.transform(X_pool, prm) if U is None else U
        lab = np.asarray(lab)
        n = len(U)
        out = np.empty((n, prm['m'].shape[0]))
        sizes = np.bincount(lab)
        s1 = sizes[lab] == 1
        T1 = prm['T1'] if T1 is None else T1
        if s1.any():
            out[s1] = _owner.log_softmax(_owner.singleton_ll(U[s1], prm['m'], prm['lam']) / T1)
        for g in np.where(sizes >= 2)[0]:
            ii = np.where(lab == g)[0]
            out[ii] = _owner.group_logpost(U[ii], prm, T1, Tg, cfg)
        return out

    def initialize(self):
        _owner = self
        global itertools, np
        import itertools
        import numpy as np
        _owner.K = 18
        _owner.DEFAULT = {'red': 'wpca', 'd': 256, 'se_shrink': 'lw', 'sp_shrink': 'lw', 'lam_floor': 0.0001, 'n_exact': 3, 'inj': True, 'iters': 5, 'repel': True, 'sp_scale': 1.0}
        _owner._MASK = {}
        _owner.LAM_DIFF = 1000000.0

    def run(self):
        self.initialize()


In [ ]:
'''Complementary ensemble: source mathematics preserved; class layers separate responsibilities.'''

In [ ]:
class ComplementaryGroups:

    def add_member(self, key, prob, rows=None, weight=None):
        _owner = self
        '''Add weight * log(prob) to the blend (all rows, or the given row subset).'''
        w = _owner.W[key] if weight is None else weight
        lp = np.log(np.clip(prob, 1e-09, 1.0))
        if rows is None:
            _owner.LOGIT[:] += w * lp
        else:
            _owner.LOGIT[rows] += w * lp
        print(f'  + {key:20s} weight {w:.3f} on {(len(_owner.TE) if rows is None else int(np.sum(rows)))} rows  ({time.time() - _owner.T0:.0f}s)', flush=True)

    def softmax_rows(self, S):
        _owner = self
        S = S - S.max(1, keepdims=True)
        P = np.exp(S)
        return P / P.sum(1, keepdims=True)

    def full_prob(self, P_part, classes):
        _owner = self
        '''Expand a (n, len(classes)) probability block to (n, K) on the global label order.'''
        P = np.zeros((len(P_part), _owner.K))
        P[:, classes] = P_part
        return P

    def group_centre_dense(self, X, groups):
        _owner = self
        X = np.asarray(X, dtype=np.float32).copy()
        for g in np.unique(groups):
            i = groups == g
            X[i] -= X[i].mean(0)
        return X

    def self_test_rows(self):
        _owner = self
        '''Training rows of the first three prompts, grouped by their true prompt (used only when the test file is tiny).'''
        idx = np.where(np.isin(_owner.PROMPT, pd.unique(_owner.PROMPT)[:3]))[0]
        return (idx, np.unique(_owner.PROMPT[idx], return_inverse=True)[1])

    def average_linkage_groups(self, D, thresh):
        _owner = self
        '''Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh.'''
        if len(D) < 2:
            return np.ones(len(D), dtype=int)
        D = np.asarray(D, dtype=np.float64).copy()
        np.fill_diagonal(D, 0.0)
        return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')

    def content_cosine_distance(self, texts, max_features=None):
        _owner = self
        X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)
        return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)

    def mixed_groups(self, texts, E, thresh=None):
        _owner = self
        if thresh is None:
            thresh = _owner.GROUP_THRESH
        '''Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E.'''
        if len(texts) < 2:
            return np.ones(len(texts), dtype=int)
        Z = np.asarray(E, dtype=np.float64)
        Z = Z - Z.mean(0)
        Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
        D = 0.5 * (_owner.content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))
        return _owner.average_linkage_groups(D, thresh)

    def normalised_content_groups(self, texts, thresh=None):
        _owner = self
        if thresh is None:
            thresh = _owner.STYLE_GROUP_THRESH
        '''The stylegram's grouping: content distance divided by its median off-diagonal value, cut at thresh.'''
        if len(texts) < 2:
            return np.ones(len(texts), dtype=int)
        D = _owner.content_cosine_distance(texts, max_features=100000)
        med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-06)
        return _owner.average_linkage_groups(D / med, thresh)

    def eligible(self, groups, lo=None, hi=None):
        _owner = self
        if lo is None:
            lo = _owner.GROUP_MIN
        if hi is None:
            hi = _owner.GROUP_MAX
        size = np.bincount(groups)[groups]
        return (size >= lo) & (size <= hi)


In [ ]:
class ComplementaryStyle(ComplementaryGroups):

    def describe_groups(self, name, groups):
        _owner = self
        size = np.bincount(groups)[groups]
        print(f'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}', flush=True)

    def shape_view(self, text):
        _owner = self
        return re.sub('[^\\W\\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)

    def function_view(self, text):
        _owner = self
        return ' '.join((t.lower() if t.lower() in _owner.FUNCTION_WORDS or not t.isalnum() else 'CONTENT' for t in _owner.TOKEN_RE.findall(text)))

    def sparse_group_centre(self, X, groups, lo=None, hi=None):
        _owner = self
        if lo is None:
            lo = _owner.GROUP_MIN
        if hi is None:
            hi = _owner.GROUP_MAX
        '''Subtract each group's mean row (sparse); groups outside [lo, hi] are left as they are.'''
        X = csr_matrix(X)
        _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
        keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
        if not len(keep):
            return X.copy()
        col = np.full(len(cnt), -1)
        col[keep] = np.arange(len(keep))
        rows = np.flatnonzero(col[inv] >= 0)
        M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
        means = diags(1.0 / cnt[keep]) @ (M.T @ X)
        return (X - M @ means).tocsr()

    def svc_probs(self, svc, X):
        _owner = self
        return _owner.full_prob(_owner.softmax_rows(svc.decision_function(X)), svc.classes_)

    def load_extra_passages(self):
        _owner = self
        sources = [('**/ntl_extra_passages_claude.csv', 'model', None), ('**/gen_passages.csv', 'model', None), ('**/ntl_generations.csv', 'label', 'finish_reason'), ('**/ntl_proxy_generations.csv', 'label', None), ('**/ntl_generated*.csv', 'model', 'finish'), ('**/passages_clean.csv', 'model', None)]
        parts = []
        for pattern, label_col, finish_col in sources:
            for f in sorted(glob.glob('/kaggle/input/' + pattern, recursive=True)):
                d = pd.read_csv(f)
                if finish_col and finish_col in d:
                    d = d[d[finish_col].eq('stop')]
                d = d.rename(columns={label_col: 'model'})[['prompt_id', 'model', 'text']].copy()
                d['prompt_id'] = ('pv_' if f.endswith('passages_clean.csv') else '') + d['prompt_id'].astype(str)
                parts.append(d)
        if not parts:
            return pd.DataFrame(columns=['prompt_id', 'model', 'text'])
        x = pd.concat(parts, ignore_index=True)
        x = x[x['model'].isin(_owner.LABELS)].copy()
        x['text'] = x['text'].astype(str).map(lambda t: re.sub('\\s+', ' ', re.sub('^\\*\\*(.*)\\*\\*$', '\\1', t.strip())).strip())
        x = x[x['text'].str.split().str.len().between(30, 300)].drop_duplicates(subset=['model', 'text'])
        x = x.drop_duplicates(subset=['prompt_id', 'model'])
        x = x[x.groupby('prompt_id')['model'].transform('nunique') >= _owner.EXTRA_MIN_LABELS].reset_index(drop=True)
        return x

    def extra_sample_weights(self):
        _owner = self
        '''Per class: original rows keep (1 - EXTRA_SHARE) of the class weight, the class's extra rows share EXTRA_SHARE of it.'''
        n_tr, n_ex = (np.bincount(_owner.y, minlength=_owner.K).astype(float), np.bincount(_owner.EX_Y, minlength=_owner.K).astype(float))
        w_tr = np.where(n_ex[_owner.y] > 0, 1.0 - _owner.EXTRA_SHARE, 1.0)
        w_ex = _owner.EXTRA_SHARE * n_tr[_owner.EX_Y] / np.maximum(n_ex[_owner.EX_Y], 1.0)
        return np.concatenate([w_tr, w_ex])

    def fit_svc(self, X, labels, C=0.5, **kw):
        _owner = self
        return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)

    def family_gate(self, texts, labels, classes, C, sample_weight=None, balanced=False):
        _owner = self
        g = _owner.StyleGrams().fit(texts)
        svc = LinearSVC(C=C, class_weight='balanced' if balanced else None, random_state=42, max_iter=4000).fit(g.transform(texts), labels, sample_weight=sample_weight)
        assert list(svc.classes_) == list(classes)
        d = svc.decision_function(g.transform(_owner.TE))
        return np.column_stack([-d, d]) if len(classes) == 2 else d


In [ ]:
class ComplementaryGaussian(ComplementaryStyle):

    def stylometry(self, t):
        _owner = self
        words = _owner.WORD_RE.findall(t)
        low = [w.lower() for w in words]
        nw, nc = (max(len(words), 1), max(len(t), 1))
        sents = [s for s in _owner.SENT_RE.split(t.strip()) if s]
        sl = np.array([len(_owner.WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
        wl = np.array([len(w) for w in words] or [0], dtype=float)
        f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw, sum((c.isupper() for c in t)) / nc, sum((c.isdigit() for c in t)) / nc, sum((ord(c) > 127 for c in t)) / nc, t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw, sum((1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they'))) / max(len(sents), 1), sum((w.endswith('ly') for w in words)) / nw, sum((w.endswith('ing') for w in words)) / nw, sum((w.endswith('tion') for w in words)) / nw]
        f += [t.count(q) / nw for q in _owner.PUNCT]
        counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
        f += [counts.get(s, 0) / nw for s in _owner.STOPS]
        return f

    def _lse(self, A, axis=None, keepdims=False):
        _owner = self
        m = np.max(A, axis=axis, keepdims=True)
        m = np.where(np.isfinite(m), m, 0.0)
        out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
        if keepdims:
            return out
        return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

    def log_softmax(self, L):
        _owner = self
        return L - _owner._lse(L, axis=1, keepdims=True)

    def _lw_shrinkage(self, X):
        _owner = self
        '''Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered).'''
        from sklearn.covariance import ledoit_wolf_shrinkage
        return float(ledoit_wolf_shrinkage(X, assume_centered=True))

    def _shrunk(self, S, a):
        _owner = self
        d = S.shape[0]
        return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

    def _two_way(self, Z, y, pinv, G, iters=4):
        _owner = self
        '''additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds).'''
        P = np.zeros((G, Z.shape[1]))
        cnt_p = np.bincount(pinv, minlength=G).astype(float)
        cls = np.unique(y)
        mu = np.zeros((_owner.K, Z.shape[1]))
        for _ in range(iters):
            R = Z - P[pinv]
            for c in cls:
                mu[c] = R[y == c].mean(0)
            R = Z - mu[y]
            P = np.zeros_like(P)
            np.add.at(P, pinv, R)
            P /= cnt_p[:, None]
        return (mu, P, cnt_p)

    def fit(self, X, y, prompts, cfg=None):
        _owner = self
        '''fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit).'''
        c = dict(_owner.DEFAULT)
        c.update(cfg or {})
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        N, D = X.shape
        pu, pinv = np.unique(np.asarray(prompts), return_inverse=True)
        G = len(pu)
        mx = X.mean(0)
        sx = X.std(0)
        sx = np.where(sx > 1e-08, sx, 1.0)
        Xs = (X - mx) / sx
        if c['red'] == 'none':
            V = np.eye(D)
        else:
            if c['red'] == 'wpca':
                Pm = np.zeros((G, D))
                np.add.at(Pm, pinv, Xs)
                Pm /= np.bincount(pinv)[:, None]
                R = Xs - Pm[pinv]
            elif c['red'] == 'pca':
                R = Xs
            else:
                raise ValueError(c['red'])
            d = int(min(c['d'], N - G - 1, D))
            if N < D:
                w, U = np.linalg.eigh(R @ R.T)
                o = np.argsort(w)[::-1][:d]
                w, U = (np.clip(w[o], 1e-12, None), U[:, o])
                V = R.T @ U / np.sqrt(w)
            else:
                _, _, Vt = np.linalg.svd(R, full_matrices=False)
                V = Vt[:d].T
        Z = Xs @ V
        d = Z.shape[1]
        mu, P, ng = _owner._two_way(Z, y, pinv, G)
        E = Z - mu[y] - P[pinv]
        dof = max(N - G - len(np.unique(y)) + 1, 1)
        Se_raw = E.T @ E / dof
        ae = _owner._lw_shrinkage(E) if c['se_shrink'] == 'lw' else float(c['se_shrink'])
        Se = _owner._shrunk(Se_raw, ae)
        Pc = P - P.mean(0)
        Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
        ap = _owner._lw_shrinkage(Pc) if c['sp_shrink'] == 'lw' else float(c['sp_shrink'])
        tr = max(np.trace(Sp_raw), 1e-08 * np.trace(Se))
        Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)
        Sp *= c['sp_scale']
        L = np.linalg.cholesky(Se)
        Li = np.linalg.inv(L)
        M = Li @ Sp @ Li.T
        M = 0.5 * (M + M.T)
        lam, Q = np.linalg.eigh(M)
        lam = np.clip(lam, c['lam_floor'], None)
        A = Q.T @ Li
        W = V @ A.T
        return {'mx': mx, 'sx': sx, 'W': W, 'm': mu @ A.T, 'lam': lam, 'T1': 1.0, 'Tg': 1.0, 'info': {'d': int(d), 'a_e': round(ae, 4), 'a_p': round(ap, 4), 'lam_pct': np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, 'cfg': c}

    def transform(self, X, prm):
        _owner = self
        return (np.asarray(X, dtype=np.float64) - prm['mx']) / prm['sx'] @ prm['W']

    def _inj_mask(self, n, C):
        _owner = self
        if (n, C) not in _owner._MASK:
            grids = np.meshgrid(*[np.arange(C)] * n, indexing='ij')
            bad = np.zeros([C] * n, bool)
            for i, j in itertools.combinations(range(n), 2):
                bad |= grids[i] == grids[j]
            _owner._MASK[n, C] = bad
        return _owner._MASK[n, C]

    def singleton_ll(self, U, m, lam):
        _owner = self
        '''(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out).'''
        R = U[:, None, :] - m[None]
        return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

    def joint_terms(self, U, m, lam):
        _owner = self
        '''a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped).'''
        n = len(U)
        gam = lam / (1 + n * lam)
        R = U[:, None, :] - m[None]
        a = -0.5 * (R ** 2 * (1 - gam)).sum(-1)
        G = R * np.sqrt(gam)
        B = np.einsum('icd,jed->ijce', G, G)
        return (a, B)

    def joint_table(self, a, B):
        _owner = self
        '''full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij.'''
        n, C = a.shape
        Q = np.zeros([C] * n)
        for i in range(n):
            sh = [1] * n
            sh[i] = C
            Q = Q + a[i].reshape(sh)
        for i, j in itertools.combinations(range(n), 2):
            sh = [1] * n
            sh[i] = C
            sh[j] = C
            Q = Q + B[i, j].reshape(sh)
        return Q

    def exact_marginals(self, Q, T=1.0, inj=True):
        _owner = self
        '''per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T.'''
        n = Q.ndim
        C = Q.shape[0]
        Qt = Q / T
        if inj and n >= 2:
            Qt = np.where(_owner._inj_mask(n, C), -np.inf, Qt)
        lz = _owner._lse(Qt)
        out = np.empty((n, C))
        for i in range(n):
            ax = tuple((k for k in range(n) if k != i))
            out[i] = (_owner._lse(Qt, axis=ax) if ax else Qt) - lz
        return out

    def em_marginals(self, U, m, lam, T=1.0, iters=5, repel=False, q0=None):
        _owner = self
        '''cavity EM for one group (any n >= 2): returns (n, C) log q.'''
        n = len(U)
        v = lam / (1 + (n - 1) * lam)
        L = _owner.singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
        q = np.exp(_owner.log_softmax(L))
        for _ in range(iters):
            s = U - q @ m
            S = s.sum(0)
            Pm = (S[None] - s) * v
            R = U[:, None, :] - Pm[:, None, :] - m[None]
            L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
            if repel:
                lo = np.log(np.clip(1 - q, 1e-06, None))
                L = L + (lo.sum(0)[None] - lo)
            q = np.exp(_owner.log_softmax(L))
        return _owner.log_softmax(L)

    def lam_g(self, prm, cfg=None):
        _owner = self
        '''lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly.'''
        c = dict(prm.get('cfg', _owner.DEFAULT))
        c.update(cfg or {})
        return np.full_like(prm['lam'], _owner.LAM_DIFF) if c.get('diff') else prm['lam']


In [ ]:
class ComplementaryFeatures(ComplementaryGaussian):

    def group_logpost(self, U, prm, T1=None, Tg=None, cfg=None):
        _owner = self
        c = dict(prm.get('cfg', _owner.DEFAULT))
        c.update(cfg or {})
        n = len(U)
        lg = _owner.lam_g(prm, cfg)
        T1 = prm['T1'] if T1 is None else T1
        Tg = prm['Tg'] if Tg is None else Tg
        if n == 1:
            return _owner.log_softmax(_owner.singleton_ll(U, prm['m'], prm['lam']) / T1)
        if n <= c['n_exact']:
            a, B = _owner.joint_terms(U, prm['m'], lg)
            return _owner.exact_marginals(_owner.joint_table(a, B), Tg, c['inj'])
        return _owner.em_marginals(U, prm['m'], lg, Tg, c['iters'], c['repel'])

    def scores(self, X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
        _owner = self
        '''(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed).'''
        U = _owner.transform(X_pool, prm) if U is None else U
        lab = np.asarray(lab)
        n = len(U)
        out = np.empty((n, prm['m'].shape[0]))
        sizes = np.bincount(lab)
        s1 = sizes[lab] == 1
        T1 = prm['T1'] if T1 is None else T1
        if s1.any():
            out[s1] = _owner.log_softmax(_owner.singleton_ll(U[s1], prm['m'], prm['lam']) / T1)
        for g in np.where(sizes >= 2)[0]:
            ii = np.where(lab == g)[0]
            out[ii] = _owner.group_logpost(U[ii], prm, T1, Tg, cfg)
        return out

    def find_checkpoint(self, model_type, hidden=None, dir_name=None):
        pattern = f'/kaggle/input/**/{dir_name}/config.json' if dir_name else '/kaggle/input/**/config.json'
        for filename in sorted(glob.glob(pattern, recursive=True)):
            path = Path(filename)
            if not InputAssets.has_weights(path.parent):
                continue
            try:
                config = json.loads(path.read_text())
            except (OSError, ValueError):
                continue
            if dir_name or (config.get('model_type') == model_type and (hidden is None or config.get('hidden_size') == hidden)):
                return str(path.parent)
        return None

    def load_truncated_decoder(self, path, keep, dtype):
        _owner = self
        cfg = AutoConfig.from_pretrained(path)
        cfg.num_hidden_layers = keep
        if getattr(cfg, 'layer_types', None):
            cfg.layer_types = cfg.layer_types[:keep]
        kw = dict(config=cfg, dtype=dtype)
        if _owner.DEV == 'cuda':
            kw['device_map'] = 'auto'
        return AutoModel.from_pretrained(path, **kw).eval()

    def style_token_mask(self, tok):
        _owner = self
        '''Vocabulary mask of 'style' tokens: the decoded piece is an English stop word or pure punctuation.'''
        pieces = [tok.decode([i]) for i in range(len(tok))]
        return torch.tensor([p.strip().lower() in ENGLISH_STOP_WORDS or bool(re.fullmatch('\\s*[^\\w\\s]+\\s*', p)) for p in pieces])

    def pooled_states(self, model, tok, texts, parts, decoder, bs, max_len=512):
        _owner = self
        '''{'L{layer}_{pool}': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only).'''
        H = model.config.hidden_size
        out = {f'L{L}_{pool}': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}
        smask = _owner.style_token_mask(tok) if any((pool == 'style' for _, pool in parts)) else None
        dev = next(model.parameters()).device
        order = np.argsort([len(t) for t in texts])
        retried = 0
        for b0 in range(0, len(texts), bs):
            idx = order[b0:b0 + bs]
            if decoder:
                batch = [(tok.bos_token or '') + texts[i] for i in idx]
                enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)
            else:
                enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(dev)
            hs = model(**enc, output_hidden_states=True).hidden_states
            if decoder and (not all((torch.isfinite(hs[L]).all() for L, _ in parts))):
                retried += 1
                model.to(torch.bfloat16)
                hs = model(**enc, output_hidden_states=True).hidden_states
            m_all = enc['attention_mask'].float()
            for L, pool in parts:
                m = m_all * smask.to(m_all.device)[enc['input_ids']].float() if pool == 'style' else m_all
                m = m.unsqueeze(-1).to(hs[L].device)
                out[f'L{L}_{pool}'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
            del hs, enc
        if retried:
            print(f'  {retried} batches redone in bf16', flush=True)
        return {k: np.nan_to_num(v) for k, v in out.items()}

    def pseudo_parts(self, parts, dim=48):
        _owner = self
        '''SMOKE only: deterministic stand-ins for the model passes (TF-IDF SVD of the texts), same downstream code.'''
        from sklearn.decomposition import TruncatedSVD
        tf = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(_owner.TR + _owner.TE)
        out = {}
        for j, (L, pool) in enumerate(parts):
            svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(_owner.TR))
            out[f'L{L}_{pool}'] = (svd.transform(tf.transform(_owner.TR)).astype(np.float32), svd.transform(tf.transform(_owner.TE)).astype(np.float32))
        return out

    def extract_parts(self, key, parts, loader):
        _owner = self
        '''Train and test pools for every requested part; loader() returns (model, tok, decoder flag, batch size).'''
        if _owner.SMOKE:
            return _owner.pseudo_parts(parts)
        model, tok, decoder, bs = loader()
        tr, te = (_owner.pooled_states(model, tok, _owner.TR, parts, decoder, bs), _owner.pooled_states(model, tok, _owner.TE, parts, decoder, bs))
        del model, tok
        gc.collect()
        if _owner.DEV == 'cuda':
            torch.cuda.empty_cache()
        return {k: (tr[k], te[k]) for k in tr}

    def decoder_loader(self, model_type, hidden, keep):
        _owner = self

        def load():
            path = _owner.find_checkpoint(model_type, hidden)
            assert path, f'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input'
            tok = AutoTokenizer.from_pretrained(path)
            tok.padding_side = 'right'
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            return (_owner.load_truncated_decoder(path, keep, torch.float16 if _owner.DEV == 'cuda' else torch.float32), tok, True, 8)
        return load

    def encoder_loader(self, dir_name):
        _owner = self

        def load():
            path = _owner.find_checkpoint(None, dir_name=dir_name)
            assert path, f'encoder directory {dir_name} not found under /kaggle/input'
            return (AutoModel.from_pretrained(path, dtype=torch.float32).to(_owner.DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16)
        return load

    def concat_parts(self, cache, parts):
        _owner = self
        return (np.concatenate([cache[f'L{L}_{p}'][0] for L, p in parts], 1), np.concatenate([cache[f'L{L}_{p}'][1] for L, p in parts], 1))

    def plain_and_twin(self, key, Etr, Ete, C):
        _owner = self
        '''Plain head (test-side standardisation) for the blend; centred twin evidence on grouped rows.'''
        if _owner.W[key] > 0:
            sc = StandardScaler().fit(Etr)
            clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), _owner.y)
            Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-06) if len(Ete) >= 2 else sc.transform(Ete)
            _owner.add_member(key, _owner.full_prob(clf.predict_proba(Xte), clf.classes_))
        else:
            print(f'  {key}: plain head has weight 0 in the 0.563 blend, not fitted', flush=True)
        if key in _owner.TWIN_W:
            csc = StandardScaler().fit(_owner.group_centre_dense(Etr, _owner.PROMPT))
            twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(_owner.group_centre_dense(Etr, _owner.PROMPT)), _owner.y)
            if _owner.GROUP_ROWS.any():
                pc = twin.predict_proba(csc.transform(_owner.group_centre_dense(Ete[_owner.GROUP_ROWS], _owner.G_TE[_owner.GROUP_ROWS])))
                _owner.add_member(key + '_twin', _owner.full_prob(pc, twin.classes_), rows=_owner.GROUP_ROWS, weight=_owner.TWIN_W[key])
            if _owner.TINY:
                st_idx, st_g = _owner.self_test_rows()
                ps = twin.predict_proba(csc.transform(_owner.group_centre_dense(Etr[st_idx], st_g)))
                print(f'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == _owner.y[st_idx]):.2f}', flush=True)

    def gaussian_group_head(self, name, spec, Xtr, Xte):
        _owner = self
        '''Our addition: fit the shared-prompt Gaussian on the training prompts, score the grouped test rows jointly within their group.'''
        prm = _owner.fit(Xtr, _owner.y, _owner.PROMPT, dict(_owner.GAUSS_SETTING, d=spec['d']))
        prm['T1'], prm['Tg'] = (spec['T1'], spec['Tg'])
        print(f"  {name}: d {prm['info']['d']}, shrinkage e/p {prm['info']['a_e']}/{prm['info']['a_p']}, lam pct {prm['info']['lam_pct']}", flush=True)
        if _owner.GROUP_ROWS.any():
            LP = _owner.scores(Xte[_owner.GROUP_ROWS], np.unique(_owner.G_TE[_owner.GROUP_ROWS], return_inverse=True)[1], prm)
            _owner.add_member(name, np.exp(LP), rows=_owner.GROUP_ROWS, weight=spec['w'])
        if _owner.TINY:
            st_idx, st_g = _owner.self_test_rows()
            LP = _owner.scores(Xtr[st_idx], st_g, prm)
            print(f'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == _owner.y[st_idx]):.2f}', flush=True)

    def score_lora_checkpoints(self, trunk_spec, pool_layers, checkpoints):
        _owner = self
        '''One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;
    within-Claude readout -> Claude gate.'''
        from peft import PeftModel
        from safetensors.torch import load_file
        model_type, H, keep = trunk_spec
        base = _owner.find_checkpoint(model_type, H)
        assert base, f'no {model_type} base checkpoint under /kaggle/input'
        cfg = AutoConfig.from_pretrained(base)
        cfg.num_hidden_layers, cfg.use_cache = (keep, False)
        if getattr(cfg, 'layer_types', None) is not None:
            cfg.layer_types = cfg.layer_types[:keep]
        trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map='auto').eval()
        for key, marker, norms_file, anchor_file in checkpoints:
            hits = sorted(glob.glob('/kaggle/input/' + marker, recursive=True))
            if not hits:
                print(f'  SKIP {key}: checkpoint not attached ({marker})', flush=True)
                continue
            ck = pathlib.Path(os.path.dirname(hits[0]))
            print(f'[{key}] {ck}', flush=True)
            tok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
            tok.padding_side = 'right'
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
            state = load_file(str(ck / norms_file))
            norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])
            norms.load_state_dict({k[len('norms.'):]: v for k, v in state.items() if k.startswith('norms.')})
            ro = np.load(ck / 'readouts.npz')
            assert list(ro['classes']) == _owner.LABELS and list(ro['pool_layers']) == pool_layers
            dev = lm.get_input_embeddings().weight.device
            norms = norms.to(dev).float()

            def token_ids(texts):
                return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=_owner.LORA_MAX_LEN)['input_ids']]

            def features(texts, bs):
                ids = token_ids(texts)
                Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)
                order = np.argsort([len(t) for t in ids], kind='stable')
                for b0 in range(0, len(order), bs):
                    idx = order[b0:b0 + bs]
                    batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
                    with torch.autocast('cuda', dtype=torch.float16):
                        hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                    m = batch['attention_mask'].unsqueeze(-1).float()
                    m[:, 0] = 0
                    Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev)) for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()
                    del hs, batch
                return Z
            if (ck / anchor_file).exists():
                saved = np.load(ck / anchor_file)['features'].astype(np.float32)
                ours = features(_owner.TR[:3], 3)
                rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-09))
                print(f'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}', flush=True)
                assert rel < 0.02, f'{key}: our features do not match the checkpoint'
            Z = features(_owner.TE, _owner.LORA_BS)
            _owner.add_member(key, _owner.softmax_rows(Z @ ro['plain_coef'].T + ro['plain_intercept']))
            if 'claude_coef' in ro.files:
                lc = Z @ ro['claude_coef'].T + ro['claude_intercept']
                _owner.GATES['claude_lora'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)
            if key in _owner.LORA_TWIN_W and _owner.GROUP_ROWS.any():
                Zc = _owner.group_centre_dense(Z[_owner.GROUP_ROWS], _owner.G_TE[_owner.GROUP_ROWS])
                _owner.add_member(key + '_twin', _owner.softmax_rows(Zc @ ro['center_coef'].T + ro['center_intercept']), rows=_owner.GROUP_ROWS, weight=_owner.LORA_TWIN_W[key])
            trunk = lm.unload()
            del lm, Z
            gc.collect()
            torch.cuda.empty_cache()
        del trunk
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
class ComplementaryAdapters(ComplementaryFeatures):

    def token_attention_probs(self, texts, head_file, backbone, bs=8):
        _owner = self
        hits = sorted(glob.glob('/kaggle/input/**/' + head_file, recursive=True))
        assert hits, f'training-kernel output holding {head_file} not attached'
        d = torch.load(hits[0], map_location='cpu')
        assert list(d['classes']) == _owner.LABELS
        decoder = d.get('decoder', True)
        path = _owner.find_checkpoint('qwen3_5') if backbone == 'qwen3_5' else _owner.find_checkpoint(None, dir_name='deberta-v3-large')
        assert path, f'backbone {backbone} not attached'
        cfg = AutoConfig.from_pretrained(path)
        if decoder:
            tc = getattr(cfg, 'text_config', None) or cfg
            tc.num_hidden_layers = d['keep']
            if getattr(tc, 'layer_types', None):
                tc.layer_types = tc.layer_types[:d['keep']]
            lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
        else:
            lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(_owner.DEV).eval()
        tok = AutoTokenizer.from_pretrained(path)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        H = d.get('hidden', 4096)
        net = _owner.TokenAttention(_owner.K, H)
        net.load_state_dict(d['state_dict'])
        dev = lm.get_input_embeddings().weight.device
        net, mu, sd = (net.to(dev).eval(), d['mu'].float().to(dev), d['sd'].float().to(dev))
        ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
        P = np.zeros((len(texts), _owner.K))
        order = np.argsort([len(t) for t in ids])
        for b0 in range(0, len(order), bs):
            idx = order[b0:b0 + bs]
            batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
            out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
            hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None else out.language_model_outputs.hidden_states)[d['layer']]
            n = batch['attention_mask'].sum(1)
            T = max(int(n.max()) - 1, 1)
            x = torch.zeros(len(idx), T, H, device=dev)
            m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)
            for r in range(len(idx)):
                k = int(n[r]) - 1
                if k > 0:
                    x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)
                    m[r, :k] = True
            P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
            del hs, out, batch
        del lm
        gc.collect()
        torch.cuda.empty_cache()
        return P

    def ultrabert_probs(self, texts):
        _owner = self
        from transformers import AutoModelForSequenceClassification
        folds = None
        for root in glob.glob('/kaggle/input/**/ultrabert-380m-ntl', recursive=True):
            fs = [f for f in sorted(glob.glob(os.path.join(root, 'fold*'))) if os.path.exists(os.path.join(f, 'config.json')) and json.load(open(os.path.join(f, 'config.json'))).get('model_type') == 'ultrabert']
            if len(fs) == 5:
                folds = fs
                break
        assert folds, 'ultrabert-380m-ntl (5 folds) not attached'
        tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)
        ids = tok(texts, truncation=True, max_length=512)['input_ids']
        order = np.argsort([len(e) for e in ids])
        P = np.zeros((len(texts), _owner.K))
        for fp in folds:
            em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation='sdpa').to(_owner.DEV).eval()
            cols = [_owner.LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]
            for b0 in range(0, len(order), 32):
                idx = order[b0:b0 + 32]
                b = tok.pad({'input_ids': [ids[j] for j in idx]}, return_tensors='pt').to(_owner.DEV)
                P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
            del em
            gc.collect()
            torch.cuda.empty_cache()
        return P

    def configure(self):
        _owner = self
        global ENGLISH_STOP_WORDS, LinearSVC, LogisticRegression, StandardScaler, TfidfVectorizer, csr_matrix, diags, fcluster, gc, glob, hstack, json, linkage, np, os, pathlib, pd, re, squareform, time, torch, vstack
        import gc, glob, json, os, pathlib, re, time
        import numpy as np, pandas as pd
        try:
            import torch
        except ImportError:
            torch = None
        from scipy.cluster.hierarchy import fcluster, linkage
        from scipy.sparse import csr_matrix, diags, hstack, vstack
        from scipy.spatial.distance import squareform
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.svm import LinearSVC
        _owner.SMOKE = os.environ.get('NTL_SMOKE') == '1'
        _owner.T0 = time.time()
        _owner.LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
        _owner.K = len(_owner.LABELS)
        _owner.LAB2I = {l: i for i, l in enumerate(_owner.LABELS)}
        _owner.FAMILY_IDX = {f: np.array([i for i, l in enumerate(_owner.LABELS) if l.startswith(f)]) for f in ('claude', 'gpt', 'gemini', 'glm')}
        _owner.GROUP_THRESH, _owner.GROUP_LAYER, _owner.GROUP_MIN, _owner.GROUP_MAX = (0.6, 24, 2, 18)
        _owner.STYLE_GROUP_THRESH = 0.9
        _owner.W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095, 'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132, 'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17, 'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
        _owner.TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.11, 'debv3l': 0.25}
        _owner.LORA_TWIN_W = {'lora24_public': 0.15, 'lora24_claude': 0.15}
        _owner.GATE_LAM = {'claude_style': 1.5, 'claude_lora': 1.0, 'family': 0.25}
        _owner.BALANCE_ALPHA = 0.5
        _owner.EXTRA_MIN_LABELS, _owner.EXTRA_SHARE = (12, 0.5)
        _owner.GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=_owner.TWIN_W['q3_8b']), 'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=_owner.TWIN_W['glm9b']), 'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=_owner.TWIN_W['q3_4b']), 'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=_owner.TWIN_W['debv3l'])}
        _owner.GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=0.0001, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)
        _owner.LM_HEADS = [('glm9b', 'glm4', 4096, [(8, 'mean'), (3, 'mean')], [(_owner.GROUP_LAYER, 'mean')], 0.0025), ('q3_8b', 'qwen3', 4096, [(7, 'mean'), (22, 'mean')], [], 0.0025), ('q25_7b', 'qwen2', 3584, [(6, 'mean'), (20, 'mean')], [], 0.0025), ('q3_4b', 'qwen3', 2560, [(4, 'style'), (7, 'style')], [(4, 'mean')], 0.0025)]
        _owner.ENC_HEADS = [('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005), ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005), ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005)]
        _owner.LORA_RUNS = [(('qwen3', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [('lora24_public', '**/lora24/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'), ('lora24_seed7', '**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz'), ('lora24_claude', '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz')]), (('qwen2', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [('lora20_q25_instyle', '**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz')])]
        _owner.LORA_MAX_LEN, _owner.LORA_BS = (447, 8)
        _owner.TOKATTN_HEADS = [('tokattn_q35', 'tokattn_final.pt', 'qwen3_5'), ('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta')]
        if os.environ.get('NTL_DROP_Q35') == '1':
            _owner.TOKATTN_HEADS = [t for t in _owner.TOKATTN_HEADS if t[0] != 'tokattn_q35']


In [ ]:
class ComplementaryEnsemble(ComplementaryAdapters):

    def load_data(self):
        _owner = self
        if _owner.SMOKE:
            _owner._all = pd.read_csv(os.environ['NTL_DATA'] + '/train.csv')
            _owner._p = _owner._all['prompt_id'].drop_duplicates().sample(frac=1, random_state=0).tolist()
            _owner.train = _owner._all[_owner._all['prompt_id'].isin(_owner._p[:30])].reset_index(drop=True)
            _owner.test = _owner._all[_owner._all['prompt_id'].isin(_owner._p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)
            _owner.test = pd.concat([_owner.test[_owner.test['prompt_id'] == p].head(k) for p, k in zip(_owner._p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)
            if os.environ.get('NTL_TINY') == '1':
                _owner.test = _owner.test.head(3).reset_index(drop=True)
            _owner.SMOKE_Y = _owner.test['model'].map(_owner.LAB2I).to_numpy(dtype=int)
        else:
            _owner.DATA_DIR = next((p for p in [pathlib.Path('/kaggle/input/competitions/name-that-llm'), pathlib.Path('/kaggle/input/name-that-llm')] if (p / 'train.csv').exists()))
            _owner.train, _owner.test = (pd.read_csv(_owner.DATA_DIR / 'train.csv'), pd.read_csv(_owner.DATA_DIR / 'test.csv'))
            if os.environ.get('NTL_FAKE_TEST'):
                _owner.test = pd.read_parquet(os.environ['NTL_POOL']).sample(int(os.environ['NTL_FAKE_TEST']), random_state=0)[['text']].reset_index(drop=True)
                _owner.test['id'] = [f'fake{i}' for i in range(len(_owner.test))]
        _owner.test['text'] = _owner.test['text'].fillna('').astype(str)
        _owner.TR, _owner.TE = (_owner.train['text'].tolist(), _owner.test['text'].tolist())
        _owner.y = _owner.train['model'].map(_owner.LAB2I).to_numpy(dtype=int)
        _owner.PROMPT = _owner.train['prompt_id'].to_numpy(dtype=object)
        _owner.TINY = len(_owner.TE) < 50
        _owner.DEV = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
        print(f'train {_owner.train.shape}  test {_owner.test.shape}  gpus {(torch.cuda.device_count() if torch is not None else 0)}  smoke {_owner.SMOKE}', flush=True)
        _owner.LOGIT = np.zeros((len(_owner.TE), _owner.K))
        _owner.G_TE, _owner.GROUP_ROWS = (None, None)

    def prepare_style(self):
        _owner = self
        _owner.FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
        _owner.TOKEN_RE = re.compile('\\w+|[^\\w\\s]')

        class StyleGrams:
            '''Four stacked TF-IDF views: char 3-5, word 1-3, shape (char 2-6 of the x/X text), function-word (word 1-3 of the masked text).'''
            VIEWS = [('char', str, (3, 5), 250000, 1.0), ('word', str, (1, 3), 180000, 0.7), ('char', _owner.shape_view, (2, 6), 60000, 0.5), ('word', _owner.function_view, (1, 3), 50000, 0.5)]

            def fit(self, texts):
                self.vecs = []
                for analyzer, view, ngr, maxf, w in self.VIEWS:
                    opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)
                    if analyzer == 'word':
                        opts['token_pattern'] = '(?u)\\b\\w+\\b|[^\\w\\s]'
                    self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))
                return self

            def transform(self, texts):
                return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format='csr')
        _owner.StyleGrams = StyleGrams
        _owner.EXTRA = _owner.load_extra_passages()
        _owner.EX_T, _owner.EX_Y, _owner.EX_G = (_owner.EXTRA['text'].tolist(), _owner.EXTRA['model'].map(_owner.LAB2I).to_numpy(dtype=int), _owner.EXTRA['prompt_id'].to_numpy(dtype=object))
        print(f"extra passages: {len(_owner.EXTRA)} rows, {_owner.EXTRA['prompt_id'].nunique()} prompts", flush=True)
        _owner.grams = _owner.StyleGrams().fit(_owner.TR + _owner.EX_T)
        _owner.X_tr, _owner.X_ex, _owner.X_te = (_owner.grams.transform(_owner.TR), _owner.grams.transform(_owner.EX_T) if len(_owner.EX_T) else None, _owner.grams.transform(_owner.TE))

    def fit_style(self):
        _owner = self
        if _owner.X_ex is not None:
            _owner.svc_plain = _owner.fit_svc(vstack([_owner.X_tr, _owner.X_ex]).tocsr(), np.concatenate([_owner.y, _owner.EX_Y]), sample_weight=_owner.extra_sample_weights())
        else:
            _owner.svc_plain = _owner.fit_svc(_owner.X_tr, _owner.y)
        _owner.P_style = _owner.svc_probs(_owner.svc_plain, _owner.X_te)
        _owner.X_tr_c = _owner.sparse_group_centre(_owner.X_tr, _owner.PROMPT)
        if _owner.X_ex is not None:
            _owner.svc_centred = _owner.fit_svc(vstack([_owner.X_tr_c, _owner.sparse_group_centre(_owner.X_ex, _owner.EX_G, _owner.GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([_owner.y, _owner.EX_Y]), sample_weight=_owner.extra_sample_weights())
        else:
            _owner.svc_centred = _owner.fit_svc(_owner.X_tr_c, _owner.y)
        _owner.G_style = _owner.normalised_content_groups(_owner.TE)
        _owner.rows_style = _owner.eligible(_owner.G_style)
        _owner.describe_groups('stylegram groups', _owner.G_style)
        if _owner.rows_style.any():
            _owner.P_style[_owner.rows_style] = _owner.svc_probs(_owner.svc_centred, _owner.sparse_group_centre(_owner.X_te[_owner.rows_style], _owner.G_style[_owner.rows_style]))
        if _owner.TINY:
            _owner.st_idx, _owner.st_g = _owner.self_test_rows()
            _owner.acc = np.mean(_owner.svc_centred.classes_[_owner.svc_centred.decision_function(_owner.sparse_group_centre(_owner.X_tr[_owner.st_idx], _owner.st_g)).argmax(1)] == _owner.y[_owner.st_idx])
            print(f'  centred stylegram self-test on {len(_owner.st_idx)} training rows (seen in training): acc {_owner.acc:.2f}', flush=True)
        _owner.add_member('stylegram', _owner.P_style)
        del _owner.X_tr_c, _owner.svc_plain, _owner.svc_centred, _owner.P_style
        _owner.GATES = {}
        _owner.api_claude = sorted(glob.glob('/kaggle/input/**/ntl_extra_passages_claude.csv', recursive=True))
        if _owner.api_claude:
            _owner.api = pd.read_csv(_owner.api_claude[0])
            _owner.m = np.isin(_owner.y, _owner.FAMILY_IDX['claude'])
            _owner.GATES['claude'] = _owner.family_gate([t for t, k in zip(_owner.TR, _owner.m) if k] + _owner.api['text'].astype(str).tolist(), np.concatenate([_owner.y[_owner.m], _owner.api['model'].map(_owner.LAB2I).to_numpy(dtype=int)]), _owner.FAMILY_IDX['claude'], 0.5)
            print(f'claude gate: {int(_owner.m.sum())} official + {len(_owner.api)} API rows', flush=True)
            del _owner.api
        else:
            print('Claude API passages not attached: Claude gate off', flush=True)
        for _owner.fam, _owner.C in [('gpt', 1.0), ('gemini', 1.0), ('glm', 0.5)]:
            _owner.m, _owner.me = (np.isin(_owner.y, _owner.FAMILY_IDX[_owner.fam]), np.isin(_owner.EX_Y, _owner.FAMILY_IDX[_owner.fam]))
            _owner.GATES[_owner.fam] = _owner.family_gate([t for t, k in zip(_owner.TR, _owner.m) if k] + [t for t, k in zip(_owner.EX_T, _owner.me) if k], np.concatenate([_owner.y[_owner.m], _owner.EX_Y[_owner.me]]), _owner.FAMILY_IDX[_owner.fam], _owner.C, sample_weight=np.concatenate([np.ones(int(_owner.m.sum())), np.full(int(_owner.me.sum()), 0.25)]), balanced=True)
            print(f'{_owner.fam} gate: {int(_owner.m.sum())} official + {int(_owner.me.sum())} extra rows', flush=True)
        del _owner.grams, _owner.X_tr, _owner.X_ex, _owner.X_te

    def fit_stylometry(self):
        _owner = self
        global lgb
        _owner.PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
        _owner.STOPS = sorted(ENGLISH_STOP_WORDS)
        _owner.SENT_RE, _owner.WORD_RE = (re.compile('(?<=[.!?])\\s+'), re.compile("[A-Za-z’']+"))
        _owner.F_tr, _owner.F_te = (np.array([_owner.stylometry(t) for t in txt], dtype=np.float32) for txt in (_owner.TR, _owner.TE))
        try:
            import lightgbm as lgb
            _owner.booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(_owner.F_tr, _owner.y)
            _owner.add_member('stylometry_lgb', _owner.full_prob(_owner.booster.predict_proba(_owner.F_te), _owner.booster.classes_))
            del _owner.booster
        except ImportError:
            assert _owner.SMOKE, 'lightgbm missing'
            print('smoke: lightgbm unavailable, stylometry member skipped', flush=True)
        del _owner.F_tr, _owner.F_te

    def initialize_gaussian(self):
        _owner = self
        global itertools
        import itertools
        _owner.K = 18
        _owner.DEFAULT = {'red': 'wpca', 'd': 256, 'se_shrink': 'lw', 'sp_shrink': 'lw', 'lam_floor': 0.0001, 'n_exact': 3, 'inj': True, 'iters': 5, 'repel': True, 'sp_scale': 1.0}
        _owner._MASK = {}
        _owner.LAM_DIFF = 1000000.0

    def score_decoders(self):
        _owner = self
        global AutoConfig, AutoModel, AutoTokenizer
        if not _owner.SMOKE:
            from transformers import AutoConfig, AutoModel, AutoTokenizer
        _owner.CACHED_PARTS = {}
        for _owner.key, _owner.model_type, _owner.hidden, _owner.parts, _owner.extra_parts, _owner.C in _owner.LM_HEADS:
            print(f'[{_owner.key}] {_owner.model_type} hidden {_owner.hidden} parts {_owner.parts + _owner.extra_parts}', flush=True)
            _owner.cache = _owner.extract_parts(_owner.key, _owner.parts + _owner.extra_parts, _owner.decoder_loader(_owner.model_type, _owner.hidden, max((L for L, _ in _owner.parts + _owner.extra_parts)) + 1))
            if _owner.G_TE is None:
                _owner.G_TE = _owner.mixed_groups(_owner.TE, _owner.cache[f'L{_owner.GROUP_LAYER}_mean'][1])
                _owner.GROUP_ROWS = _owner.eligible(_owner.G_TE)
                _owner.describe_groups('test groups (content + GLM L24)', _owner.G_TE)
                if _owner.SMOKE:
                    _owner.same_g, _owner.same_p = (_owner.G_TE[:, None] == _owner.G_TE[None, :], _owner.test['prompt_id'].to_numpy(dtype=object)[:, None] == _owner.test['prompt_id'].to_numpy(dtype=object)[None, :])
                    _owner.off = ~np.eye(len(_owner.TE), dtype=bool)
                    print(f'  smoke grouping: pair precision {(_owner.same_g & _owner.same_p & _owner.off).sum() / max((_owner.same_g & _owner.off).sum(), 1):.2f} recall {(_owner.same_g & _owner.same_p & _owner.off).sum() / max((_owner.same_p & _owner.off).sum(), 1):.2f}', flush=True)
            _owner.CACHED_PARTS[_owner.key] = _owner.cache
            _owner.plain_and_twin(_owner.key, *_owner.concat_parts(_owner.cache, _owner.parts), _owner.C)
            for _owner.name, _owner.spec in _owner.GAUSS_HEADS.items():
                if _owner.spec['src'] == _owner.key:
                    _owner.gaussian_group_head(_owner.name, _owner.spec, *_owner.cache[_owner.spec['part']])
            del _owner.cache
            _owner.CACHED_PARTS.pop(_owner.key)
            gc.collect()

    def score_encoders(self):
        _owner = self
        for _owner.key, _owner.dir_name, _owner.parts, _owner.C in _owner.ENC_HEADS:
            print(f'[{_owner.key}] {_owner.dir_name} parts {_owner.parts}', flush=True)
            _owner.cache = _owner.extract_parts(_owner.key, _owner.parts, _owner.encoder_loader(_owner.dir_name))
            _owner.plain_and_twin(_owner.key, *_owner.concat_parts(_owner.cache, _owner.parts), _owner.C)
            for _owner.name, _owner.spec in _owner.GAUSS_HEADS.items():
                if _owner.spec['src'] == _owner.key:
                    _owner.gaussian_group_head(_owner.name, _owner.spec, *_owner.cache[_owner.spec['part']])
            del _owner.cache
            gc.collect()

    def prepare_lora(self):
        _owner = self

    def score_lora(self):
        _owner = self
        if not _owner.SMOKE:
            for _owner.trunk_spec, _owner.pool_layers, _owner.checkpoints in _owner.LORA_RUNS:
                _owner.score_lora_checkpoints(_owner.trunk_spec, _owner.pool_layers, _owner.checkpoints)

    def prepare_token_attention(self):
        _owner = self

        class TokenAttention(torch.nn.Module if torch is not None else object):

            def __init__(self, k, h=4096):
                super().__init__()
                self.drop, self.proj, self.score, self.out = (torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k))

            def forward(self, x, m):
                h = torch.tanh(self.proj(self.drop(x)))
                a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -10000.0), -1).unsqueeze(-1)
                mf = m.unsqueeze(-1).float()
                return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))
        _owner.TokenAttention = TokenAttention

    def score_token_attention(self):
        _owner = self
        if not _owner.SMOKE:
            for _owner.key, _owner.head_file, _owner.backbone in _owner.TOKATTN_HEADS:
                if _owner.TINY:
                    _owner.acc = float(np.mean(_owner.token_attention_probs(_owner.TR[:40], _owner.head_file, _owner.backbone).argmax(1) == _owner.y[:40]))
                    print(f'  {_owner.key} in-sample check on 40 training rows: acc {_owner.acc:.2f}', flush=True)
                    assert _owner.acc > 0.5, f'{_owner.key} does not reproduce its training fit'
                _owner.add_member(_owner.key, _owner.token_attention_probs(_owner.TE, _owner.head_file, _owner.backbone))
            _owner.add_member('ultrabert', _owner.ultrabert_probs(_owner.TE))
        np.save('/kaggle/working/comp_logit.npy', _owner.LOGIT)
        json.dump({'labels': _owner.LABELS, 'ids': _owner.test['id'].astype(str).tolist(), 'seconds': time.time() - _owner.T0}, open('/kaggle/working/comp_meta.json', 'w'))
        print(f'COMP STACK DONE {time.time() - _owner.T0:.0f}s', flush=True)

    @torch.inference_mode()
    def run(self):
        self.configure()
        self.load_data()
        self.prepare_style()
        self.fit_style()
        self.fit_stylometry()
        self.initialize_gaussian()
        self.score_decoders()
        self.score_encoders()
        self.prepare_lora()
        self.score_lora()
        self.prepare_token_attention()
        self.score_token_attention()


In [ ]:
'''Main signature ensemble: source mathematics preserved; class layers separate responsibilities.'''

In [ ]:
class SignatureUtilities:

    def __init__(self, argv):
        self.argv = argv

    def log(self, *x):
        _owner = self
        print(f'[{time.time() - _owner.T0:7.0f}s]', *x, flush=True)

    def center_groups(self, X, groups):
        _owner = self
        X = X.copy()
        for g in np.unique(groups):
            i = groups == g
            X[i] -= X[i].mean(0)
        return X

    def torch_lr(self, Xtr, ytr, Xte, C, iters=200):
        _owner = self
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        Xtr = torch.as_tensor(Xtr, dtype=torch.float32, device=dev)
        Xte = torch.as_tensor(Xte, dtype=torch.float32, device=dev)
        mu, sd = (Xtr.mean(0), Xtr.std(0) + 1e-06)
        Xtr = (Xtr - mu) / sd
        Xte = (Xte - mu) / sd
        yt = torch.as_tensor(ytr, dtype=torch.long, device=dev)
        W = torch.zeros(Xtr.shape[1], 18, device=dev, requires_grad=True)
        b = torch.zeros(18, device=dev, requires_grad=True)
        opt = torch.optim.LBFGS([W, b], lr=1, max_iter=iters, history_size=20, line_search_fn='strong_wolfe')

        def closure():
            opt.zero_grad()
            loss = torch.nn.functional.cross_entropy(Xtr @ W + b, yt, reduction='sum') + 0.5 / C * (W ** 2).sum()
            loss.backward()
            return loss
        opt.step(closure)
        with torch.no_grad():
            return torch.log_softmax(Xte @ W + b, -1).cpu().numpy()

    def capped_agglo(self, S, cap=18, thr=0.02):
        _owner = self
        '''Average linkage (Lance-Williams), never merging past `cap` members; stop when best avg sim < thr.'''
        n = len(S)
        A = S.astype(np.float32).copy()
        np.fill_diagonal(A, -np.inf)
        size = np.ones(n, np.int32)
        alive = np.ones(n, bool)
        lab = np.arange(n)
        while True:
            ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
            M = np.where(ok, A, -np.inf)
            k = int(np.argmax(M))
            i, j = divmod(k, n)
            if not np.isfinite(M[i, j]) or M[i, j] < thr:
                break
            new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
            A[i] = new
            A[:, i] = new
            A[i, i] = -np.inf
            A[j] = -np.inf
            A[:, j] = -np.inf
            size[i] += size[j]
            alive[j] = False
            lab[lab == j] = i
        return lab


In [ ]:
class SignatureDecoders(SignatureUtilities):

    def hungarian(self, lp, groups):
        _owner = self
        pred = lp.argmax(1).copy()
        for g in np.unique(groups):
            idx = np.where(groups == g)[0]
            r, c = linear_sum_assignment(-lp[idx])
            pred[idx[r]] = c
        return pred

    @torch.no_grad()
    def embed(self, spec, texts):
        _owner = self
        tok = AutoTokenizer.from_pretrained(spec['path'])
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        kw = dict(device_map='auto')
        q = spec.get('quant', 'fp16')
        if q == '4bit':
            from transformers import BitsAndBytesConfig
            kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
        elif q == '8bit':
            from transformers import BitsAndBytesConfig
            kw['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)
        kw['dtype'] = torch.bfloat16 if q == 'bf16' else torch.float16
        m = AutoModel.from_pretrained(spec['path'], **kw).eval()
        enc = tok(list(texts), truncation=True, max_length=512)['input_ids']
        order = np.argsort([len(x) for x in enc])
        out = None
        bs = spec.get('bs', 16)
        nan_rows = 0
        first = next(m.parameters()).device
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            L = max((len(enc[j]) for j in b))
            ids = torch.full((len(b), L), tok.pad_token_id)
            att = torch.zeros((len(b), L), dtype=torch.long)
            for r, j in enumerate(b):
                ids[r, :len(enc[j])] = torch.tensor(enc[j])
                att[r, :len(enc[j])] = 1
            hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True).hidden_states
            nL = len(hs) - 1
            sel = spec['layers_abs'] if 'layers_abs' in spec else [max(1, round(f * nL)) for f in spec['layers']]
            pooled = []
            for l in sel:
                h = hs[l].float()
                w = att.to(h.device).unsqueeze(-1).float()
                pooled.append(((h * w).sum(1) / w.sum(1)).cpu())
            pooled = torch.cat(pooled, 1).numpy()
            nan_rows += int((~np.isfinite(pooled)).any(1).sum())
            if out is None:
                out = np.zeros((len(texts), pooled.shape[1]), np.float32)
                _owner.log('  layers', sel, 'of', nL)
            out[b] = pooled
        del m
        gc.collect()
        torch.cuda.empty_cache()
        if nan_rows:
            _owner.log('  WARNING non-finite rows:', nan_rows)
        return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    @torch.no_grad()
    def lora_predict(self, spec, texts):
        _owner = self
        '''LoRA classifier WITHOUT peft: merge W += (B @ A) * alpha/r into the base weights, load the trained score head.'''
        from transformers import AutoModelForSequenceClassification
        from safetensors.torch import load_file
        tok = AutoTokenizer.from_pretrained(spec['path'])
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        m = AutoModelForSequenceClassification.from_pretrained(spec['path'], num_labels=18, dtype=torch.float16, device_map='auto')
        m.config.pad_token_id = tok.pad_token_id
        cfg = json.load(open(os.path.join(spec['adapter'], 'adapter_config.json')))
        scale = cfg['lora_alpha'] / cfg['r']
        sd = load_file(os.path.join(spec['adapter'], 'adapter_model.safetensors'))
        mods = dict(m.named_modules())
        n_merged = 0
        head = None
        for k, v in sd.items():
            key = k.replace('base_model.model.', '', 1)
            if '.lora_A.' in key:
                mod_name = key.split('.lora_A.')[0]
                B = sd[k.replace('.lora_A.', '.lora_B.')]
                lin = mods[mod_name]
                W = lin.weight
                W += (B.to(W.device, torch.float32) @ v.to(W.device, torch.float32) * scale).to(W.dtype)
                n_merged += 1
            elif key.startswith('score.') and 'weight' in key and ('original_module' not in key):
                head = v
        assert head is not None and n_merged > 0, (n_merged, [k for k in sd if 'score' in k])
        m.score.weight.copy_(head.to(m.score.weight.device, m.score.weight.dtype))
        _owner.log(f'  merged {n_merged} LoRA pairs (scale {scale}), head {tuple(head.shape)}')
        del mods, sd, lin, W, B, head
        gc.collect()
        m.eval()
        enc = tok(list(texts), truncation=True, max_length=512)['input_ids']
        order = np.argsort([len(x) for x in enc])
        out = np.zeros((len(texts), 18), np.float32)
        bs = spec.get('bs', 16)
        first = next(m.parameters()).device
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            L = max((len(enc[j]) for j in b))
            ids = torch.full((len(b), L), tok.pad_token_id)
            att = torch.zeros((len(b), L), dtype=torch.long)
            for r, j in enumerate(b):
                ids[r, :len(enc[j])] = torch.tensor(enc[j])
                att[r, :len(enc[j])] = 1
            lo = m(input_ids=ids.to(first), attention_mask=att.to(first)).logits.float()
            out[b] = torch.log_softmax(lo, -1).cpu().numpy()
        del m
        gc.collect()
        torch.cuda.empty_cache()
        return out


In [ ]:
class SignatureModels(SignatureDecoders):

    @torch.no_grad()
    def l24_predict(self, spec, texts, groups=None, train_texts=None):
        _owner = self
        '''LoRA24-recipe member(s): truncated base loaded ONCE; for each adapter dir in spec['adapters'] (or spec['adapter'])
    merge W += B A * s, compute masked mean-pooled multi-layer features, read plain (+ centred) heads, then unmerge.
    Returns log-probs averaged over adapters (plain, centred-or-None).'''
        from transformers import AutoConfig
        from safetensors.torch import load_file
        adapters = spec.get('adapters') or [spec['adapter']]
        tok = AutoTokenizer.from_pretrained(spec['path'])
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        cfg = AutoConfig.from_pretrained(spec['path'])
        tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
        tc.num_hidden_layers = spec['keep']
        tc.use_cache = False
        if getattr(tc, 'layer_types', None) is not None:
            tc.layer_types = tc.layer_types[:spec['keep']]
        m = AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto').eval()
        if spec.get('no_final_norm'):
            m.norm = torch.nn.Identity()
        mods = dict(m.named_modules())
        first = next(m.parameters()).device
        L = spec['layers']
        _eos = spec.get('eos_id')
        _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get('max_len', 384) - (1 if _eos is not None else 0))['input_ids']]
        enc = _tk(texts)
        order = np.argsort([len(x) for x in enc])
        bs = spec.get('bs', 16)

        def lsm(z):
            return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
        P_acc, C_acc = (0.0, 0.0)
        FEATS = None
        OVF = [0]
        if train_texts is not None:
            enc_tr = _tk(train_texts)

        def featurize(enc_, norms, hid):
            order_ = np.argsort([len(x) for x in enc_])
            Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)
            for i in range(0, len(order_), bs):
                b = order_[i:i + bs]
                Lm = max((len(enc_[j]) for j in b))
                ids = torch.full((len(b), Lm), tok.pad_token_id)
                att = torch.zeros((len(b), Lm), dtype=torch.long)
                for r, j in enumerate(b):
                    ids[r, :len(enc_[j])] = torch.tensor(enc_[j])
                    att[r, :len(enc_[j])] = 1
                hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
                if not all((torch.isfinite(hs[l]).all() for l in L)):
                    m.to(torch.bfloat16)
                    hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
                    m.to(torch.float16)
                    OVF[0] += 1
                mask = att.unsqueeze(-1).float()
                mask[:, 0] = 0
                Z_[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()
            return Z_
        for ai, ad in enumerate(adapters):
            adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
            acfg = json.load(open(os.path.join(adp, 'adapter_config.json')))
            scale = acfg['lora_alpha'] / acfg['r']
            sd = load_file(os.path.join(adp, 'adapter_model.safetensors'))
            deltas = []
            for k, v in sd.items():
                if '.lora_A.' not in k:
                    continue
                name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
                if name not in mods and 'model.' + name in mods:
                    name = 'model.' + name
                W = mods[name].weight
                B = sd[k.replace('.lora_A.', '.lora_B.')]
                W += (B.to(W.device, torch.float32) @ v.to(W.device, torch.float32) * scale).to(W.dtype)
                deltas.append((W, B, v))
            if os.path.exists(os.path.join(ad, 'heads.pt')):
                H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
            else:
                H = dict(load_file(os.path.join(ad, 'heads.safetensors')))
            if os.path.exists(os.path.join(ad, 'meta.json')):
                _lab = json.load(open(os.path.join(ad, 'meta.json')))['labels']
                _perm = [_lab.index(l) for l in _owner.LABELS]
                for _p in ('plain', 'centered'):
                    H[f'{_p}.weight'] = H[f'{_p}.weight'][_perm]
                    H[f'{_p}.bias'] = H[f'{_p}.bias'][_perm]
            hid = H['norms.0.weight'].shape[0]
            norms = [torch.nn.LayerNorm(hid) for _ in L]
            for i, nm in enumerate(norms):
                nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
            Z = np.zeros((len(texts), hid * len(L)), np.float32)
            for i in range(0, len(order), bs):
                b = order[i:i + bs]
                Lm = max((len(enc[j]) for j in b))
                ids = torch.full((len(b), Lm), tok.pad_token_id)
                att = torch.zeros((len(b), Lm), dtype=torch.long)
                for r, j in enumerate(b):
                    ids[r, :len(enc[j])] = torch.tensor(enc[j])
                    att[r, :len(enc[j])] = 1
                hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
                if not all((torch.isfinite(hs[l]).all() for l in L)):
                    m.to(torch.bfloat16)
                    hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
                    m.to(torch.float16)
                    OVF[0] += 1
                mask = att.unsqueeze(-1).float()
                mask[:, 0] = 0
                Z[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()
            if train_texts is not None and ai == len(adapters) - 1:
                FEATS = (Z.copy(), featurize(enc_tr, norms, hid))
            for W, B, A_ in deltas:
                W -= (B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32) * scale).to(W.dtype)
            torch.cuda.empty_cache()
            del deltas, sd
            gc.collect()
            hd = lambda p, X: lsm(X @ H[f'{p}.weight'].float().numpy().T + H[f'{p}.bias'].float().numpy())
            P_acc = P_acc + hd('plain', Z) / len(adapters)
            if groups is not None:
                Zc = Z.copy()
                for g in np.unique(groups):
                    ii = groups == g
                    Zc[ii] -= Zc[ii].mean(0)
                C_acc = C_acc + hd('centered', Zc) / len(adapters)
            _owner.log(f"  l24 {spec['name']}: adapter {os.path.basename(ad)} done (bf16 redo batches {OVF[0]})")
        del m, mods
        gc.collect()
        torch.cuda.empty_cache()
        if train_texts is not None:
            return (P_acc, C_acc if groups is not None else None, FEATS)
        return (P_acc, C_acc if groups is not None else None)

    def gpu_mem(self):
        _owner = self
        return ' '.join((f'gpu{i}:{torch.cuda.memory_allocated(i) / 1000000000.0:.1f}GB' for i in range(torch.cuda.device_count())))

    def atlas_scores(self, bundle, rows, budget):
        _owner = self
        '''Source-conditioned likelihood probabilities (public ATLAS bundle format) for test rows `rows`, in our label order.
    Rows are scored shortest-first; raises if the token-proportional projection would overrun `budget` (s since start).'''
        sys.path.insert(0, bundle)
        for _m in [m_ for m_ in list(sys.modules) if m_.startswith(('ntl_atlas', 'ntl_rlcd'))]:
            del sys.modules[_m]
        from ntl_atlas.model import load_model as _atlas_load
        from ntl_rlcd.data import LABELS as _alab
        from safetensors.torch import load_file as _lf
        sys.path.remove(bundle)
        _cfg = json.load(open(os.path.join(bundle, 'atlas_bundle.json')))['training_config']
        _am, _atok = _atlas_load(_owner.a.atlas_base, _alab, _cfg, device='auto')
        try:
            _am.load_trainable(_lf(os.path.join(bundle, 'adapters.safetensors')))
            _am.eval()
            _enc = _atok(_owner.test.text.values[rows].tolist(), truncation=True, max_length=_cfg['max_length'])['input_ids']
            _ord = np.argsort([len(t) for t in _enc], kind='stable')
            out = np.zeros((len(rows), 18), np.float32)
            _t1 = time.time()
            _ctok = np.cumsum(np.array([len(_enc[int(i)]) for i in _ord], np.float64))
            for _b in range(0, len(_ord), _owner.a.atlas_bs):
                _idx = _ord[_b:_b + _owner.a.atlas_bs]
                _pk = _atok.pad({'input_ids': [_enc[int(i)] for i in _idx]}, padding=True, return_tensors='pt').to(_am.device)
                with torch.inference_mode():
                    _p, _ = _am.score(_pk['input_ids'], _pk['attention_mask'], source_chunk=_owner.a.atlas_chunk)
                out[_idx] = _p.float().cpu().numpy()
                _done = _b + len(_idx)
                if _b % (25 * _owner.a.atlas_bs) == 0:
                    _proj = time.time() - _owner.T0 + (time.time() - _t1) / _ctok[_done - 1] * (_ctok[-1] - _ctok[_done - 1]) * 1.1
                    _owner.log(f'atlas {os.path.basename(bundle)} {_done}/{len(_ord)} projected finish {_proj:.0f}s')
                    if _done > 50 and _proj > budget:
                        raise RuntimeError('atlas would overrun the time budget')
                    if _owner.a.atlas_probe and _done >= _owner.a.atlas_probe:
                        raise RuntimeError(f'atlas probe stop; projected {_proj:.0f}s')
            assert np.isfinite(out).all() and np.allclose(out.sum(1), 1, atol=0.0001)
            return out[:, [_alab.index(l) for l in _owner.LABELS]]
        finally:
            del _am
            gc.collect()
            torch.cuda.empty_cache()


In [ ]:
class SignatureEnsemble(SignatureModels):

    def configure(self):
        _owner = self
        global argparse, gc, json, linear_sum_assignment, np, os, pd, sp, sys, time, torch
        import os, sys, time, gc, argparse, json
        import numpy as np, pandas as pd, scipy.sparse as sp, torch
        from scipy.optimize import linear_sum_assignment
        _owner.LABELS = ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max']
        _owner.L2I = {l: i for i, l in enumerate(_owner.LABELS)}
        _owner.p = argparse.ArgumentParser()
        _owner.p.add_argument('--data', default='/kaggle/input/competitions/name-that-llm')
        _owner.p.add_argument('--models', default='')
        _owner.p.add_argument('--comp_path', default='')
        _owner.p.add_argument('--comp_w', type=float, default=0.0)
        _owner.p.add_argument('--fake_test', type=int, default=0, help='timing probe: score N pool passages as a stand-in test')
        _owner.p.add_argument('--atlas_probe', type=int, default=0, help='timing probe: stop ATLAS after N passages')
        _owner.p.add_argument('--sim', type=str, default='', help='comma list of folds held out of train.csv as fake test')
        _owner.p.add_argument('--out', default='submission.parquet')
        _owner.p.add_argument('--cache', default='cache')
        _owner.p.add_argument('--dtype', default='fp16')
        _owner.p.add_argument('--C', type=float, default=0.005)
        _owner.p.add_argument('--tfidf_w', type=float, default=1.0)
        _owner.p.add_argument('--distort_w', type=float, default=0.0, help='weight of topic-masked (text distortion) TF-IDF member')
        _owner.p.add_argument('--adapt_thr', type=float, default=0.15, help='cluster cut (avg cosine sim on content-word TF-IDF)')
        _owner.p.add_argument('--adapt_w', type=float, default=0.0, help='weight of centred twin evidence on rows in clusters of size>=2 (0=off)')
        _owner.p.add_argument('--post', type=int, default=0, help='1: spaced em-dash guard + Hungarian on dense test clusters (12-18)')
        _owner.p.add_argument('--atlas_dir', default='')
        _owner.p.add_argument('--atlas_base', default='')
        _owner.p.add_argument('--atlas_w', default='0', help='mix weight per bundle (comma list or one value)')
        _owner.p.add_argument('--atlas_margin', type=float, default=0.0, help='>0: score ATLAS only on rows whose top-2 margin is below this')
        _owner.p.add_argument('--atlas_bs', type=int, default=4)
        _owner.p.add_argument('--atlas_chunk', type=int, default=3)
        _owner.p.add_argument('--atlas_budget', type=float, default=8.3 * 3600)
        _owner.p.add_argument('--sk_div', type=int, default=1, help='1: divide blend by #members before Sinkhorn (ours); 0: raw blend (public notebook)')
        _owner.p.add_argument('--sinkhorn', type=float, default=0.0, help='alpha of Sinkhorn class-marginal balancing (0=off)')
        _owner.p.add_argument('--sim_k', default='', help="sim only: keep 'lo-hi' random passages per held-out prompt")
        _owner.p.add_argument('--extra', default='', help='parquet/csv (text, model) of matched external passages, used by TF-IDF/distort members')
        _owner.p.add_argument('--extra_w', type=float, default=0.5)
        _owner.p.add_argument('--svc_w', type=float, default=0.0, help='4-view n-gram LinearSVC on train + --pool')
        _owner.p.add_argument('--pool', default='')
        _owner.p.add_argument('--save_lp', default='', help='also save normalised blend log-probs (n x 18, LABELS order) to this .npy')
        _owner.p.add_argument('--perpassage', type=int, default=1, help='1: test passages treated independently (no clustering/centring/assignment)')
        _owner.a, _owner._ = _owner.p.parse_known_args(self.argv)
        os.makedirs(_owner.a.cache, exist_ok=True)
        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
        _owner.T0 = time.time()

    def load_data(self):
        _owner = self
        _owner.train = pd.read_csv(os.path.join(_owner.a.data, 'train.csv'))
        if _owner.a.sim:
            _owner.rng = np.random.RandomState(0)
            _owner.pids = np.array(sorted(_owner.train.prompt_id.unique()))
            _owner.rng.shuffle(_owner.pids)
            _owner.pf = {q: i % 5 for i, q in enumerate(_owner.pids)}
            _owner.fo = _owner.train.prompt_id.map(_owner.pf).values
            _owner.hold = np.isin(_owner.fo, [int(x) for x in _owner.a.sim.split(',')])
            if _owner.a.sim_k:
                _owner.lo_, _owner.hi_ = map(int, _owner.a.sim_k.split('-'))
                _owner.keep = np.zeros(len(_owner.train), bool)
                _owner.idx_h = np.where(_owner.hold)[0]
                for _owner.g in np.unique(_owner.train.prompt_id.values[_owner.idx_h]):
                    _owner.ii = _owner.idx_h[_owner.train.prompt_id.values[_owner.idx_h] == _owner.g]
                    _owner.keep[_owner.rng.choice(_owner.ii, min(len(_owner.ii), _owner.rng.randint(_owner.lo_, _owner.hi_ + 1)), replace=False)] = True
                _owner.hold = _owner.keep | _owner.hold & False
            _owner.test = _owner.train[_owner.hold].drop(columns=['model']).reset_index(drop=True)
            _owner.test_truth = _owner.train[_owner.hold].reset_index(drop=True)
            _owner.hold_any = np.isin(_owner.fo, [int(x) for x in _owner.a.sim.split(',')])
            _owner.train = _owner.train[~_owner.hold_any].reset_index(drop=True)
        elif _owner.a.fake_test:
            _owner.test = pd.read_parquet(_owner.a.pool).sample(_owner.a.fake_test, random_state=0)[['text']].reset_index(drop=True)
            _owner.test['id'] = [f'fake{i}' for i in range(len(_owner.test))]
        else:
            _owner.test = pd.read_csv(os.path.join(_owner.a.data, 'test.csv'))
        _owner.train['text'] = _owner.train.text.fillna('')
        _owner.test['text'] = _owner.test.text.fillna('')
        _owner.y = _owner.train.model.map(_owner.L2I).values
        _owner.Gtr = _owner.train.prompt_id.values
        _owner.log('train', len(_owner.train), 'test', len(_owner.test))

    def recover_groups(self):
        _owner = self
        global TfidfVectorizer, adjusted_rand_score
        from sklearn.feature_extraction.text import TfidfVectorizer
        if _owner.a.perpassage and _owner.a.adapt_w <= 0:
            _owner.Gte = np.arange(len(_owner.test))
        else:
            try:
                _owner.v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
                _owner.Xc = _owner.v.fit_transform(_owner.test.text.values)
                _owner.Gte = _owner.capped_agglo((_owner.Xc @ _owner.Xc.T).toarray(), 18, _owner.a.adapt_thr if _owner.a.perpassage else 0.02)
            except ValueError as e:
                _owner.e = e
                _owner.log('clustering fallback:', _owner.e)
                _owner.Gte = np.arange(len(_owner.test))
        _owner.log('test clusters', len(np.unique(_owner.Gte)), 'sizes', np.bincount(np.bincount(_owner.Gte)[np.bincount(_owner.Gte) > 0]).tolist())
        if _owner.a.sim:
            from sklearn.metrics import adjusted_rand_score
            _owner.log('cluster ARI vs truth', round(adjusted_rand_score(_owner.test_truth.prompt_id, _owner.Gte), 4))

    def prepare_text(self):
        _owner = self
        global LogisticRegression
        _owner.LP = {}
        _owner.EXT_T, _owner.EXT_Y = (np.array([], dtype=object), np.array([], dtype=int))
        if _owner.a.extra:
            _owner._E = pd.read_parquet(_owner.a.extra) if _owner.a.extra.endswith('.parquet') else pd.read_csv(_owner.a.extra)
            _owner._E = _owner._E[_owner._E.model.isin(_owner.LABELS)]
            _owner.EXT_T = _owner._E.text.astype(str).values
            _owner.EXT_Y = _owner._E.model.map(_owner.L2I).values
            _owner.log('extra rows', len(_owner.EXT_T))
        _owner.FIT_T = np.concatenate([_owner.train.text.values, _owner.EXT_T])
        _owner.FIT_Y = np.concatenate([_owner.y, _owner.EXT_Y])
        _owner.FIT_W = np.concatenate([np.ones(len(_owner.train)), np.full(len(_owner.EXT_T), _owner.a.extra_w)])
        from sklearn.linear_model import LogisticRegression
        if _owner.a.tfidf_w > 0:
            _owner.mats_tr, _owner.mats_te = ([], [])
            for _owner.kw in [dict(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=60000), dict(token_pattern='\\w+|[^\\w\\s]', ngram_range=(1, 2), min_df=3, max_features=30000)]:
                _owner.vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **_owner.kw).fit(_owner.FIT_T if _owner.a.perpassage else _owner.train.text.values)
                _owner.mats_tr.append(_owner.vec.transform(_owner.FIT_T if _owner.a.perpassage else _owner.train.text.values))
                _owner.mats_te.append(_owner.vec.transform(_owner.test.text.values))
            if _owner.a.perpassage:
                _owner.LP['tfidf'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(_owner.mats_tr).tocsr(), _owner.FIT_Y, sample_weight=_owner.FIT_W).predict_log_proba(sp.hstack(_owner.mats_te).tocsr()) * _owner.a.tfidf_w
            else:
                _owner.Xtr = _owner.center_groups(sp.hstack(_owner.mats_tr).toarray().astype(np.float32), _owner.Gtr)
                _owner.Xte = _owner.center_groups(sp.hstack(_owner.mats_te).toarray().astype(np.float32), _owner.Gte)
                _owner.LP['tfidf'] = LogisticRegression(C=5, max_iter=3000).fit(_owner.Xtr, _owner.y).predict_log_proba(_owner.Xte) * _owner.a.tfidf_w
                del _owner.Xtr, _owner.Xte
        gc.collect()
        _owner.log('tfidf done')

    def fit_svc(self):
        _owner = self
        global AutoModel, AutoTokenizer, LinearSVC, _hs, _re, collections, re
        if _owner.a.svc_w > 0:
            import re as _re
            from scipy.sparse import hstack as _hs
            from sklearn.svm import LinearSVC
            _owner._FW = set('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
            _owner._shape = lambda t: _re.sub('[^\\W\\d_]+', lambda m_: 'X' if m_[0].isupper() else 'x', t)
            _owner._func = lambda t: ' '.join((w.lower() if w.lower() in _owner._FW or not w.isalnum() else 'CONTENT' for w in _re.findall('\\w+|[^\\w\\s]', t)))
            _owner._P = pd.read_parquet(_owner.a.pool)
            _owner._P = _owner._P[_owner._P.model.isin(_owner.LABELS)]
            _owner._fit = list(_owner.train.text.values) + list(_owner._P.text.astype(str).values)
            _owner._fy = np.r_[_owner.y, _owner._P.model.map(_owner.L2I).values]
            _owner._A, _owner._B = ([], [])
            for _owner.view, _owner.lo, _owner.hi, _owner.mf, _owner.wv in [('char', 3, 5, 250000, 1.0), ('word', 1, 3, 180000, 0.7), ('shape', 2, 6, 60000, 0.5), ('function', 1, 3, 50000, 0.5)]:
                _owner.prep = _owner._shape if _owner.view == 'shape' else _owner._func if _owner.view == 'function' else lambda s_: s_
                _owner.an = 'char' if _owner.view == 'shape' else 'word' if _owner.view == 'function' else _owner.view
                _owner.o = dict(analyzer=_owner.an, ngram_range=(_owner.lo, _owner.hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=_owner.mf, dtype=np.float32)
                if _owner.an == 'word':
                    _owner.o['token_pattern'] = '(?u)\\b\\w+\\b|[^\\w\\s]'
                _owner._v = TfidfVectorizer(**_owner.o)
                _owner._A.append(_owner._v.fit_transform([_owner.prep(t) for t in _owner._fit]) * _owner.wv)
                _owner._B.append(_owner._v.transform([_owner.prep(t) for t in _owner.test.text.values]) * _owner.wv)
            _owner._d = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced').fit(_hs(_owner._A, format='csr'), _owner._fy).decision_function(_hs(_owner._B, format='csr'))
            _owner.LP['svc4'] = (_owner._d - _owner._d.max(1, keepdims=True) - np.log(np.exp(_owner._d - _owner._d.max(1, keepdims=True)).sum(1, keepdims=True))) * _owner.a.svc_w
            del _owner._A, _owner._B
            gc.collect()
            _owner.log('svc4 done')
        if _owner.a.distort_w > 0:
            import re, collections
            _owner.wre = re.compile("[A-Za-z’']+")
            _owner.cnt = collections.Counter((w.lower() for t in _owner.FIT_T for w in _owner.wre.findall(t)))
            _owner.keep = {w for w, _ in _owner.cnt.most_common(100)}
            _owner.dist = lambda arr: [_owner.wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in _owner.keep else '*', t) for t in arr]
            _owner.Dtr, _owner.Dte = (_owner.dist(_owner.FIT_T), _owner.dist(_owner.test.text.values))
            _owner.vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2), TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern='[^\\s]+', ngram_range=(1, 3), min_df=2)]
            _owner.Xa = sp.hstack([v_.fit_transform(_owner.Dtr) for v_ in _owner.vs]).tocsr()
            _owner.Xb = sp.hstack([v_.transform(_owner.Dte) for v_ in _owner.vs]).tocsr()
            _owner.LP['distort'] = LogisticRegression(C=10, max_iter=3000).fit(_owner.Xa, _owner.FIT_Y, sample_weight=_owner.FIT_W).predict_log_proba(_owner.Xb) * _owner.a.distort_w
            _owner.log('distort done')
        from transformers import AutoTokenizer, AutoModel
        _owner.TPL = 'Identify which AI language model wrote the following passage, based only on its writing style.\n\nPassage:\n{}\n\nThe author model is:'

    def score_models(self):
        _owner = self
        global _lf
        _owner.specs = json.loads(_owner.a.models) if _owner.a.models else []
        _owner.all_text = np.concatenate([_owner.train.text.values, _owner.test.text.values])
        for _owner.spec in _owner.specs:
            if _owner.spec.get('type') == 'l24':
                _owner.log('l24', _owner.spec['name'])
                if len(_owner.test) < 50:
                    _owner._p40, _owner._ = _owner.l24_predict(_owner.spec, _owner.train.text.values[:40])
                    _owner._acc = float((_owner._p40.argmax(1) == _owner.y[:40]).mean())
                    _owner.log(f"  {_owner.spec['name']} in-sample check acc {_owner._acc:.2f}")
                    assert _owner._acc > 0.5, f"{_owner.spec['name']} l24 does not reproduce its fit"
                if _owner.spec.get('gauss', 0) > 0:
                    _owner.plain, _owner.cent, (_owner.Zte_g, _owner.Ztr_g) = _owner.l24_predict(_owner.spec, _owner.test.text.values, _owner.Gte if _owner.a.adapt_w > 0 else None, train_texts=_owner.train.text.values)
                    _ga = SharedPromptGaussian()
                    _ga.initialize()
                    _owner.prm = _ga.fit(_owner.Ztr_g, _owner.y, _owner.Gtr, dict(red='wpca', d=128, se_shrink='lw', sp_shrink='lw', lam_floor=0.0001, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))
                    _owner.prm['T1'] = _owner.prm['Tg'] = 3.0
                    _owner._sz = np.bincount(_owner.Gte)[_owner.Gte]
                    _owner._rows = (_owner._sz >= 2) & (_owner._sz <= 18)
                    _owner._q = _ga.scores(_owner.Ztr_g[[0, 1]], np.zeros(2, dtype=int), _owner.prm)
                    assert np.allclose(np.exp(_owner._q).sum(1), 1)
                    _owner._G = np.zeros((len(_owner.test), 18))
                    if _owner._rows.any():
                        _owner._G[_owner._rows] = _ga.scores(_owner.Zte_g[_owner._rows], np.unique(_owner.Gte[_owner._rows], return_inverse=True)[1], _owner.prm)
                    _owner.LP[_owner.spec['name'] + '_gauss'] = _owner._G * _owner.spec['gauss']
                    _owner.log(f"  gaussian head {_owner.spec['name']}: {int(_owner._rows.sum())} grouped test rows, weight {_owner.spec['gauss']}")
                else:
                    _owner.plain, _owner.cent = _owner.l24_predict(_owner.spec, _owner.test.text.values, _owner.Gte if _owner.a.adapt_w > 0 else None)
                _owner.LP[_owner.spec['name']] = _owner.plain * _owner.spec.get('w', 1.0)
                if _owner.cent is not None:
                    _owner.rows = np.bincount(_owner.Gte)[_owner.Gte] >= 2
                    _owner.LP[_owner.spec['name'] + '_twin'] = np.where(_owner.rows[:, None], _owner.cent, 0.0) * _owner.a.adapt_w * _owner.spec.get('w', 1.0) * _owner.spec.get('twin', 1.0)
                gc.collect()
                torch.cuda.empty_cache()
                _owner.log(_owner.spec['name'], 'done', _owner.gpu_mem())
                continue
            if _owner.spec.get('type') == 'enc':
                _owner.log('enc', _owner.spec['name'])
                from safetensors.torch import load_file as _lf
                _owner._ck = _owner.spec['ckpt']
                _owner._meta = json.load(open(os.path.join(_owner._ck, 'meta.json')))
                _owner._perm = [_owner._meta['labels'].index(l) for l in _owner.LABELS]
                _owner._hd = _lf(os.path.join(_owner._ck, 'head.safetensors'))
                _owner._W = _owner._hd['weight'].float().numpy()[_owner._perm]
                _owner._b = _owner._hd['bias'].float().numpy()[_owner._perm]
                _owner._etok = AutoTokenizer.from_pretrained(os.path.join(_owner._ck, 'encoder'))
                _owner._enc = AutoModel.from_pretrained(os.path.join(_owner._ck, 'encoder'), dtype=torch.float32).to('cuda:0').eval()

                def _enc_feats(texts):
                    _ids = _owner._etok(list(texts), truncation=True, max_length=_owner._meta['max_len'])['input_ids']
                    _ord = np.argsort([len(x) for x in _ids], kind='stable')
                    Z_ = np.zeros((len(_ids), _owner._enc.config.hidden_size), np.float32)
                    with torch.inference_mode():
                        for _s in range(0, len(_ord), 32):
                            _idx = _ord[_s:_s + 32]
                            _bt = _owner._etok.pad({'input_ids': [_ids[i] for i in _idx]}, return_tensors='pt').to('cuda:0')
                            _m = _bt['attention_mask'].unsqueeze(-1).float()
                            Z_[_idx] = ((_owner._enc(**_bt).last_hidden_state * _m).sum(1) / _m.sum(1).clamp(min=1)).cpu().numpy()
                    return Z_
                _owner._enc_feats = _enc_feats
                if len(_owner.test) < 50:
                    _owner._acc = float(((_owner._enc_feats(_owner.train.text.values[:40]) @ _owner._W.T + _owner._b).argmax(1) == _owner.y[:40]).mean())
                    _owner.log(f"  {_owner.spec['name']} in-sample check acc {_owner._acc:.2f}")
                    assert _owner._acc > 0.5, _owner.spec['name']
                _owner._zz = _owner._enc_feats(_owner.test.text.values) @ _owner._W.T + _owner._b
                _owner.LP[_owner.spec['name']] = (_owner._zz - _owner._zz.max(1, keepdims=True) - np.log(np.exp(_owner._zz - _owner._zz.max(1, keepdims=True)).sum(1, keepdims=True))) * _owner.spec.get('w', 1.0)
                del _owner._enc
                gc.collect()
                torch.cuda.empty_cache()
                _owner.log(_owner.spec['name'], 'done', _owner.gpu_mem())
                continue
            if _owner.spec.get('type') == 'head':
                _owner.log('head', _owner.spec['name'])
                _owner.H = np.load(_owner.spec['head'])
                _owner.Ete = _owner.embed(dict(_owner.spec, layers=sorted((float(v) for v in _owner.H['layers']))), _owner.test.text.values)
                if len(_owner.test) < 50:
                    _owner._E40 = _owner.embed(dict(_owner.spec, layers=sorted((float(v) for v in _owner.H['layers']))), _owner.train.text.values[:40])
                    _owner._acc = float((((_owner._E40 - _owner.H['mu']) / _owner.H['sd'] @ _owner.H['W'] + _owner.H['b']).argmax(1) == _owner.y[:40]).mean())
                    _owner.log(f"  {_owner.spec['name']} in-sample check acc {_owner._acc:.2f}")
                    assert _owner._acc > 0.5, f"{_owner.spec['name']} head does not reproduce its fit"
                _owner.Z = (_owner.Ete - _owner.H['mu']) / _owner.H['sd']
                _owner.zz = _owner.Z @ _owner.H['W'] + _owner.H['b']
                _owner.LP[_owner.spec['name']] = (_owner.zz - _owner.zz.max(1, keepdims=True) - np.log(np.exp(_owner.zz - _owner.zz.max(1, keepdims=True)).sum(1, keepdims=True))) * _owner.spec.get('w', 1.0)
                gc.collect()
                torch.cuda.empty_cache()
                _owner.log(_owner.spec['name'], 'done', _owner.gpu_mem())
                continue
            if _owner.spec.get('type') == 'lora':
                _owner.log('lora', _owner.spec['name'])
                _owner.LP[_owner.spec['name']] = _owner.lora_predict(_owner.spec, _owner.test.text.values) * _owner.spec.get('w', 1.0)
                gc.collect()
                torch.cuda.empty_cache()
                _owner.log(_owner.spec['name'], 'done', _owner.gpu_mem())
                continue
            _owner.name = _owner.spec['name']
            _owner.cp = os.path.join(_owner.a.cache, f"{_owner.name}_{('sim' + _owner.a.sim + _owner.a.sim_k if _owner.a.sim else 'real')}.npy")
            if os.path.exists(_owner.cp):
                _owner.E = np.load(_owner.cp)
            else:
                _owner.log('embedding', _owner.name)
                _owner.E = _owner.embed(_owner.spec, _owner.all_text)
                np.save(_owner.cp, _owner.E)
            _owner.Etr, _owner.Ete = (_owner.E[:len(_owner.train)], _owner.E[len(_owner.train):])
            _owner.r = _owner.torch_lr(_owner.Etr, _owner.y, _owner.Ete, _owner.spec.get('C', _owner.a.C))
            if _owner.a.perpassage:
                _owner.LP[_owner.name] = _owner.r * _owner.spec.get('w', 1.0)
                if _owner.a.adapt_w > 0:
                    _owner.sizes = np.bincount(_owner.Gte)[_owner.Gte]
                    _owner.rows = _owner.sizes >= 2
                    _owner.c = _owner.torch_lr(_owner.center_groups(_owner.Etr, _owner.Gtr), _owner.y, _owner.center_groups(_owner.Ete, _owner.Gte), _owner.spec.get('C', _owner.a.C))
                    _owner.LP[_owner.name + '_twin'] = np.where(_owner.rows[:, None], _owner.c, 0.0) * _owner.a.adapt_w * _owner.spec.get('w', 1.0)
            else:
                _owner.c = _owner.torch_lr(_owner.center_groups(_owner.Etr, _owner.Gtr), _owner.y, _owner.center_groups(_owner.Ete, _owner.Gte), _owner.a.C)
                _owner.LP[_owner.name] = (_owner.r + _owner.c) * _owner.spec.get('w', 1.0)
            gc.collect()
            torch.cuda.empty_cache()
            _owner.log(_owner.name, 'done', _owner.gpu_mem())

    def collect_evidence(self):
        _owner = self
        _owner.GUARDS = [(' — ', {'claude-opus-4-6', 'claude-opus-5', 'glm-5.3', 'glm-5.3-flash'}), ('’', {l for l in _owner.LABELS if not l.startswith(('claude', 'glm'))}), ('"', {'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'kimi-k3'}), ('“', {'gemini-3.8-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'}), ('–', {'claude-opus-5', 'gemini-3.1-pro-preview', 'glm-5.3-flash', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'kimi-k3'})]
        _owner.tot = sum(_owner.LP.values())

    def blend_and_balance(self):
        _owner = self
        _owner.COMP_P = None
        if _owner.a.comp_w > 0 and _owner.a.comp_path and os.path.exists(_owner.a.comp_path):
            try:
                _owner._cm = json.load(open(_owner.a.comp_path.replace('comp_logit.npy', 'comp_meta.json')))
                assert _owner._cm['ids'] == _owner.test['id'].astype(str).tolist(), 'comp stack row order differs'
                _owner._cl = np.load(_owner.a.comp_path)[:, [_owner._cm['labels'].index(l) for l in _owner.LABELS]]
                _owner.COMP_P = np.exp(_owner._cl - _owner._cl.max(1, keepdims=True))
                _owner.COMP_P /= _owner.COMP_P.sum(1, keepdims=True)
                assert np.isfinite(_owner.COMP_P).all()
                _owner.log(f"comp stack loaded ({_owner._cm['seconds']:.0f}s)")
            except Exception as _e:
                _owner._e = _e
                _owner.log('COMP SKIPPED:', repr(_owner._e))
                _owner.COMP_P = None
        elif _owner.a.comp_w > 0:
            _owner.log('COMP SKIPPED: no stack output')
        _owner._bundles = [d for d in _owner.a.atlas_dir.split(',') if d]
        _owner._aw = [float(x) for x in str(_owner.a.atlas_w).split(',')] if _owner._bundles else []
        if _owner._bundles and len(_owner._aw) == 1:
            _owner._aw = _owner._aw * len(_owner._bundles)
        if _owner.a.sinkhorn > 0 and len(_owner.test) >= 18 or _owner._bundles or _owner.COMP_P is not None:
            _owner.wsum = sum((abs(v).max() > 0 for v in _owner.LP.values()))
            _owner.lpn = _owner.tot / max(1, _owner.wsum) if _owner.a.sk_div else _owner.tot
            _owner.P = np.exp(_owner.lpn - _owner.lpn.max(1, keepdims=True))
            _owner.P /= _owner.P.sum(1, keepdims=True)
            if _owner.COMP_P is not None:
                _owner.P = (1 - _owner.a.comp_w) * _owner.P + _owner.a.comp_w * _owner.COMP_P
                _owner.P /= _owner.P.sum(1, keepdims=True)
            if _owner._bundles and sum(_owner._aw) > 0:
                _owner._s = np.sort(_owner.P, 1)
                _owner._margin = _owner._s[:, -1] - _owner._s[:, -2]
                _owner.rows = np.where(_owner._margin < _owner.a.atlas_margin)[0] if _owner.a.atlas_margin > 0 and len(_owner.test) >= 50 else np.arange(len(_owner.test))
                _owner.log(f'atlas on {len(_owner.rows)}/{len(_owner.test)} rows (top-2 margin < {_owner.a.atlas_margin})')
                _owner.mixed = _owner.P.copy()
                _owner._wleft = 1.0
                for _owner._bd, _owner._w in zip(_owner._bundles, _owner._aw):
                    try:
                        _owner.A_ = _owner.atlas_scores(_owner._bd, _owner.rows, _owner.a.atlas_budget)
                        _owner.mixed[_owner.rows] = _owner.mixed[_owner.rows] - _owner._w * _owner.P[_owner.rows] + _owner._w * _owner.A_
                        _owner.log(f'atlas {_owner._bd} mixed at {_owner._w}', _owner.gpu_mem())
                    except Exception as _e:
                        _owner._e = _e
                        _owner.log('ATLAS SKIPPED:', _owner._bd, repr(_owner._e))
                _owner.P = _owner.mixed / _owner.mixed.sum(1, keepdims=True)
            _owner.Q = _owner.P.copy()
            if not (_owner.a.sinkhorn > 0 and len(_owner.test) >= 18):
                _owner.tot = np.log(_owner.P + 1e-12)
            if _owner.a.sinkhorn > 0 and len(_owner.test) >= 18:
                for _owner._ in range(100):
                    _owner.Q *= len(_owner.Q) / 18 / _owner.Q.sum(0, keepdims=True)
                    _owner.Q /= _owner.Q.sum(1, keepdims=True)
                _owner.tot = (1 - _owner.a.sinkhorn) * np.log(_owner.P + 1e-12) + _owner.a.sinkhorn * np.log(_owner.Q + 1e-12)

    def apply_source_rules(self):
        _owner = self
        if _owner.a.post and _owner.a.perpassage:
            _owner._nfix = 0
            for _owner._mark, _owner._allowed in _owner.GUARDS:
                _owner._forb = np.array([l not in _owner._allowed for l in _owner.LABELS])
                for _owner._i, _owner._t in enumerate(_owner.test.text.values):
                    if _owner._mark in _owner._t and _owner.LABELS[int(_owner.tot[_owner._i].argmax())] not in _owner._allowed:
                        _owner.tot[_owner._i, _owner._forb] = -1000000000.0
                        _owner._nfix += 1
            _owner._sz = np.bincount(_owner.Gte)[_owner.Gte]
            _owner._nh = 0
            for _owner._g in np.unique(_owner.Gte[(_owner._sz >= 12) & (_owner._sz <= 18)]):
                _owner._ii = np.where(_owner.Gte == _owner._g)[0]
                _owner._r, _owner._c = linear_sum_assignment(-_owner.tot[_owner._ii])
                _owner._nh += int((_owner.tot[_owner._ii].argmax(1)[_owner._r] != _owner._c).sum())
                _owner.tot[_owner._ii[_owner._r]] = -1000000000.0
                _owner.tot[_owner._ii[_owner._r], _owner._c] = 0.0
            _owner.log(f'post: em-dash guard fixed {_owner._nfix}, dense-cluster Hungarian changed {_owner._nh}')

    def write_submission(self):
        _owner = self
        global _lsm, bas
        _owner.pred = _owner.tot.argmax(1) if _owner.a.perpassage else _owner.hungarian(_owner.tot, _owner.Gte)
        if _owner.a.sim:
            from sklearn.metrics import balanced_accuracy_score as bas
            _owner.yt = _owner.test_truth.model.map(_owner.L2I).values
            for _owner.k, _owner.v_ in _owner.LP.items():
                _owner.log(f'  {_owner.k:12s} arg {bas(_owner.yt, _owner.v_.argmax(1)):.4f} hung {bas(_owner.yt, _owner.hungarian(_owner.v_, _owner.Gte)):.4f}')
            _owner.log(f'ENSEMBLE arg {bas(_owner.yt, _owner.tot.argmax(1)):.4f} | hung(est clusters) {bas(_owner.yt, _owner.pred):.4f} | hung(true prompts) {bas(_owner.yt, _owner.hungarian(_owner.tot, _owner.test_truth.prompt_id.values)):.4f}')
            np.save(os.path.join(_owner.a.cache, f'sim{_owner.a.sim}_lp.npy'), _owner.tot)
        if _owner.a.save_lp:
            _owner.wtot = sum((abs(float(s_.get('w', 1.0))) for s_ in _owner.specs)) + (_owner.a.tfidf_w if _owner.a.tfidf_w > 0 else 0) + (_owner.a.distort_w if _owner.a.distort_w > 0 else 0) + (_owner.a.svc_w if _owner.a.svc_w > 0 else 0)
            from scipy.special import log_softmax as _lsm
            np.save(_owner.a.save_lp, _lsm(_owner.tot / max(_owner.wtot, 1e-06), axis=1).astype(np.float32))
            _owner.log('saved', _owner.a.save_lp, 'norm', _owner.wtot)
        _owner.sub = pd.DataFrame({'id': _owner.test.id.values, 'model': [_owner.LABELS[i] for i in _owner.pred]})
        assert _owner.sub.model.isin(_owner.LABELS).all() and _owner.sub.id.is_unique and (len(_owner.sub) == len(_owner.test))
        _owner.sub.to_parquet(_owner.a.out, index=False)
        _owner.log('wrote', _owner.a.out, _owner.sub.model.value_counts().to_dict())

    @torch.inference_mode()
    def run(self):
        self.configure()
        self.load_data()
        self.recover_groups()
        self.prepare_text()
        self.fit_svc()
        self.score_models()
        self.collect_evidence()
        self.blend_and_balance()
        self.apply_source_rules()
        self.write_submission()


In [ ]:
'''Research changes are isolated in one subclass; all main weights and public checkpoints stay fixed.'''


In [ ]:
class StableSignatureEnsemble(SignatureEnsemble):
    def __init__(self, argv, delta=0.03):
        super().__init__(argv)
        self.delta = delta

    def capped_agglo(self, S, cap=18, thr=0.02):
        base = super().capped_agglo(S, cap, thr)
        lower = super().capped_agglo(S, cap, thr - self.delta)
        upper = super().capped_agglo(S, cap, thr + self.delta)
        result, audit = StableGroupPolicy.retain(base, lower, upper, cap)
        self.group_audit = dict(audit, threshold=thr, delta=self.delta)
        self.source_groups = base.copy()
        print('STABLE GROUPS:', json.dumps(self.group_audit), flush=True)
        return result


In [ ]:
class ExperimentRunner:
    def __init__(self, config, assets):
        self.config = config
        self.assets = assets

    @staticmethod
    def worker(kind, config, assets, argv):
        '''Each GPU phase owns one child process, so all its CUDA allocations disappear on exit.'''
        logfile = config.output_root / f'{kind}.log'
        with logfile.open('w', buffering=1) as stream, redirect_stdout(stream), redirect_stderr(stream):
            torch.set_num_threads(4)
            os.environ['NTL_DROP_Q35'] = '1'
            os.environ.pop('NTL_SMOKE', None)
            os.environ.pop('NTL_FAKE_TEST', None)
            RuntimeCompatibility.probe()
            if kind == 'complement':
                AdapterHeadChecks.run(assets)
            if kind == 'complement':
                engine = ComplementaryEnsemble()
            else:
                engine = StableSignatureEnsemble(argv, config.group_delta) if config.stable_groups else SignatureEnsemble(argv)
            with RuntimeCompatibility.dense_peft(), threadpool_limits(limits=4):
                engine.run()
            if kind == 'signature':
                np.savez_compressed(config.output_root / 'main_evidence.npz', ids=engine.test.id.to_numpy(dtype=str),
                                    labels=np.asarray(engine.LABELS, dtype=str), groups=engine.Gte,
                                    source_groups=getattr(engine, 'source_groups', engine.Gte),
                                    **{f'log_evidence_{key}': value for key, value in engine.LP.items()})
                probabilities = np.exp(engine.tot - engine.tot.max(1, keepdims=True))
                probabilities /= probabilities.sum(1, keepdims=True)
                np.savez_compressed(config.output_root / 'final_scores.npz', ids=engine.test.id.to_numpy(dtype=str), labels=np.asarray(engine.LABELS, dtype=str), probabilities=probabilities)
                report = {'research': config.stable_groups, 'group_audit': getattr(engine, 'group_audit', None),
                          'note': 'Final scores include source typography rules and Hungarian assignment. They are not calibrated confidence estimates.',
                          'source': SOURCE_URL, 'source_sha256': SOURCE_SHA256, 'omitted_source_members': ['ll8v5']}
                (config.output_root / 'experiment_audit.json').write_text(json.dumps(report, indent=2))

    def phase(self, kind, argv, deadline):
        context = multiprocessing.get_context('fork')
        process = context.Process(target=self.worker, args=(kind, self.config, self.assets, argv))
        start = time.monotonic()
        process.start()
        logfile = self.config.output_root / f'{kind}.log'
        consumed = 0
        try:
            while process.is_alive():
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    raise TimeoutError(f'{kind} exceeded the notebook wall-time budget; see {logfile}')
                process.join(timeout=min(15, remaining))
                if logfile.exists():
                    with logfile.open() as stream:
                        stream.seek(consumed)
                        update = stream.read()
                        consumed = stream.tell()
                    if update:
                        print(update, end='', flush=True)
            process.join()
            if logfile.exists():
                with logfile.open() as stream:
                    stream.seek(consumed)
                    print(stream.read(), end='', flush=True)
            if process.exitcode != 0:
                raise RuntimeError(f'{kind} failed with exit code {process.exitcode}. See {logfile}; no fallback submission will be used.')
            return time.monotonic() - start
        finally:
            if process.is_alive():
                process.terminate()
                process.join(timeout=10)
                if process.is_alive():
                    process.kill()
                    process.join()

    def install_offline_wheels(self):
        wheels = [p for p in self.assets.files if p.name.startswith('bitsandbytes-') and p.suffix == '.whl']
        if len(wheels) > 1:
            raise ValueError('Multiple bitsandbytes wheels attached; keep the source offline-wheel input only')
        if wheels:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--quiet', str(wheels[0])], check=True)

    def run(self):
        started = time.monotonic()
        deadline = started + self.config.total_budget_seconds
        root = self.config.output_root
        # A failed rerun must not leave a submission produced by a previous run.
        for name in ['submission.parquet', 'candidate_submission.parquet', 'comp_logit.npy', 'comp_meta.json']:
            (root / name).unlink(missing_ok=True)
        self.install_offline_wheels()
        previous = Path.cwd()
        try:
            os.chdir(root)
            comp_seconds = self.phase('complement', [], deadline)
            if not (root / 'comp_logit.npy').is_file() or not (root / 'comp_meta.json').is_file():
                raise FileNotFoundError('Complementary worker did not produce its required output')
            atlas_budget = int(8.3 * 3600 - comp_seconds - 600)
            if atlas_budget <= 0:
                raise TimeoutError('Complementary phase consumed the source ATLAS budget')
            values = {'data': self.assets.data, 'models': json.dumps(self.assets.models()), 'perpassage': 1,
                      'extra': self.assets.extra, 'extra_w': 0.5, 'tfidf_w': 0, 'distort_w': 0, 'svc_w': 1.5,
                      'pool': self.assets.pool, 'adapt_w': 1.0, 'adapt_thr': 0.15, 'sinkhorn': 1.0, 'post': 1,
                      'atlas_dir': self.assets.atlas, 'atlas_base': self.assets.b8, 'atlas_w': 0.10, 'atlas_margin': 0.3,
                      'atlas_bs': 4, 'atlas_chunk': 3, 'atlas_budget': atlas_budget,
                      'comp_path': str(root / 'comp_logit.npy'), 'comp_w': 0.15,
                      'out': str(root / 'candidate_submission.parquet'), 'cache': str(root / 'cache')}
            argv = [part for key, value in values.items() for part in ['--' + key, str(value)]]
            main_seconds = self.phase('signature', argv, deadline)
            submission = pd.read_parquet(root / 'candidate_submission.parquet')
            test = pd.read_csv(Path(self.assets.data) / 'test.csv')
            if list(submission.columns) != ['id', 'model'] or submission.id.tolist() != test.id.tolist():
                raise ValueError('Submission schema or test ID order differs')
            if not submission.model.isin(LABELS).all() or submission.model.isna().any():
                raise ValueError('Submission contains invalid labels')
            (root / 'candidate_submission.parquet').replace(root / 'submission.parquet')
            text = '\n'.join((root / f'{kind}.log').read_text() for kind in ['complement', 'signature'])
            skips = [line for line in text.splitlines() if 'SKIP' in line]
            timings = {'complement_seconds': comp_seconds, 'signature_seconds': main_seconds,
                       'total_seconds': time.monotonic() - started, 'source_optional_skips': skips}
            (root / 'stage_timings.json').write_text(json.dumps(timings, indent=2))
            if any('COMP SKIPPED' in line for line in skips):
                (root / 'submission.parquet').unlink(missing_ok=True)
                raise RuntimeError('Complementary probabilities were rejected; see stage_timings.json')
            print('VALIDATED OUTPUT:', root / 'submission.parquet', submission.shape, flush=True)
            return submission
        except BaseException:
            (root / 'submission.parquet').unlink(missing_ok=True)
            raise
        finally:
            os.chdir(previous)


In [ ]:
'''CPU policy checks run before asset loading. Preflight-only mode: set run_inference=False below.'''
StableGroupPolicy.self_check()
config = ExperimentConfig()
assets = InputAssets(config).preflight()

In [ ]:
'''Run the complete experiment. Successful output is /kaggle/working/submission.parquet.'''
if config.run_inference:
    submission = ExperimentRunner(config, assets).run()
    print(submission.head())
else:
    print('PREFLIGHT ONLY: no submission was created.')

In [ ]:
'''Artifacts
submission.parquet: validated Kaggle submission, written only after all required stages succeed.
input_manifest.json: exact source hash and resolved input paths.
complement.log / signature.log: complete worker diagnostics and source optional skips.
main_evidence.npz: per-member log evidence, labels, test IDs and chosen prompt groups.
final_scores.npz: scores after source postprocessing; not calibrated probabilities.
experiment_audit.json / stage_timings.json: research policy, limitations, elapsed times and skips.
Use these ID-aligned artifacts for later CPU comparisons; no new embeddings are needed for that analysis.
'''